# Clean Sessions


Nhiem vu cua phan nay la tao du lieu sach truoc khi EDA/model, sau do tao label future 30 ngay cho bai toan moi.

Muc tieu output:
- `data_pyspark_parquet/train_sessions_clean_30d_label`
- `data_pyspark_parquet/test_sessions_clean_30d_label`

Nguyen tac quan trong:
- Khong sua truc tiep dataset goc da parse, ma tao ban clean rieng de co the trace lai quyet dinh xu ly.
- Khong tao target tu current session.
- Current session revenue/transaction chi duoc dung lam source signal de tao future label va audit, khong dung lam feature model.


### Buoc 1: Doc lai du lieu goc da parse

Muc tieu: doc lai cac bang Parquet da tao o notebook 01 va issue log da tao o notebook 02 de chuan bi cho cleaning.

Dau vao:
- `data_pyspark_parquet/train_sessions`
- `data_pyspark_parquet/test_sessions`
- `data_pyspark_parquet/data_quality_issue_log`

Viec can lam:
1. Khoi tao SparkSession va khai bao path.
2. Kiem tra folder dau vao va `_SUCCESS` marker.
3. Doc train/test sessions va data quality issue log.
4. Kiem tra nhanh so dong, so cot, schema va sample.
5. Xem nhanh cac issue can uu tien xu ly trong buoc clean tiep theo.

#### Buoc 1.1: Khoi tao SparkSession

Khoi tao Spark local, cau hinh Python environment va khai bao cac duong dan input/output dung cho notebook clean sessions.

In [1]:
import os
import sys
from pathlib import Path

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# Neu notebook chay tren Windows local va da cai Hadoop winutils.
HADOOP_HOME_PATH = Path(r"C:\hadoop")
if HADOOP_HOME_PATH.exists():
    os.environ["HADOOP_HOME"] = str(HADOOP_HOME_PATH)
    os.environ["PATH"] = str(HADOOP_HOME_PATH / "bin") + os.pathsep + os.environ.get("PATH", "")

# Dam bao Spark driver va Python worker dung cung Python executable.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "week_3":
    PROJECT_ROOT = PROJECT_ROOT.parent

PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"

TRAIN_PARQUET_PATH = PARQUET_DIR / "train_sessions"
TEST_PARQUET_PATH = PARQUET_DIR / "test_sessions"
DATA_QUALITY_ISSUE_LOG_PATH = PARQUET_DIR / "data_quality_issue_log"

TRAIN_CLEAN_PARQUET_PATH = PARQUET_DIR / "train_sessions_clean"
TEST_CLEAN_PARQUET_PATH = PARQUET_DIR / "test_sessions_clean"

spark = (
    SparkSession.builder
    .appName("week3-clean-sessions")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Project root:", PROJECT_ROOT)
print("Parquet dir:", PARQUET_DIR)
print("Train input:", TRAIN_PARQUET_PATH)
print("Test input:", TEST_PARQUET_PATH)
print("Data quality issue log:", DATA_QUALITY_ISSUE_LOG_PATH)
print("Train clean output:", TRAIN_CLEAN_PARQUET_PATH)
print("Test clean output:", TEST_CLEAN_PARQUET_PATH)

Spark version: 4.1.2
Project root: g:\ds
Parquet dir: g:\ds\data_pyspark_parquet
Train input: g:\ds\data_pyspark_parquet\train_sessions
Test input: g:\ds\data_pyspark_parquet\test_sessions
Data quality issue log: g:\ds\data_pyspark_parquet\data_quality_issue_log
Train clean output: g:\ds\data_pyspark_parquet\train_sessions_clean
Test clean output: g:\ds\data_pyspark_parquet\test_sessions_clean


#### Buoc 1.2: Kiem tra cac folder dau vao

Kiem tra cac folder Parquet co ton tai hay khong va co `_SUCCESS` marker hay khong. Neu thieu input thi dung lai ngay, vi cac buoc clean phai dua tren output cua notebook 01 va 02.

In [2]:
def check_parquet_input_path(name, path):
    return {
        "dataset": name,
        "path": str(path),
        "path_exists": path.exists(),
        "success_marker_exists": (path / "_SUCCESS").exists(),
    }


input_path_check_rows = [
    check_parquet_input_path("train_sessions", TRAIN_PARQUET_PATH),
    check_parquet_input_path("test_sessions", TEST_PARQUET_PATH),
    check_parquet_input_path("data_quality_issue_log", DATA_QUALITY_ISSUE_LOG_PATH),
]

input_path_check_df = spark.createDataFrame(input_path_check_rows)
input_path_check_df.show(truncate=False)

missing_input_paths = [row for row in input_path_check_rows if not row["path_exists"]]
if missing_input_paths:
    raise FileNotFoundError(f"Thieu input Parquet: {missing_input_paths}")

+----------------------+-------------------------------------------------+-----------+---------------------+
|dataset               |path                                             |path_exists|success_marker_exists|
+----------------------+-------------------------------------------------+-----------+---------------------+
|train_sessions        |g:\ds\data_pyspark_parquet\train_sessions        |true       |true                 |
|test_sessions         |g:\ds\data_pyspark_parquet\test_sessions         |true       |true                 |
|data_quality_issue_log|g:\ds\data_pyspark_parquet\data_quality_issue_log|true       |true                 |
+----------------------+-------------------------------------------------+-----------+---------------------+



#### Buoc 1.3: Doc train/test sessions va issue log

Doc thanh cac DataFrame dung chung cho notebook:
- `train_sessions_df`
- `test_sessions_df`
- `data_quality_issue_log_df`

In [3]:
train_sessions_df = spark.read.parquet(str(TRAIN_PARQUET_PATH))
test_sessions_df = spark.read.parquet(str(TEST_PARQUET_PATH))
data_quality_issue_log_df = spark.read.parquet(str(DATA_QUALITY_ISSUE_LOG_PATH))

print("Doc Parquet thanh cong.")
print("Train columns:", len(train_sessions_df.columns))
print("Test columns:", len(test_sessions_df.columns))
print("Issue log columns:", len(data_quality_issue_log_df.columns))

Doc Parquet thanh cong.
Train columns: 61
Test columns: 61
Issue log columns: 9


#### Buoc 1.4: Kiem tra nhanh so dong, so cot va schema

Tao bang overview de chac chan cac dataset doc duoc dung kich thuoc mong doi, sau do in schema de biet cac cot nao se xu ly trong cac buoc clean tiep theo.

In [4]:
def summarize_input_dataset(name, df, path):
    return {
        "dataset": name,
        "path": str(path),
        "row_count": df.count(),
        "column_count": len(df.columns),
        "success_marker_exists": (path / "_SUCCESS").exists(),
    }


input_overview_rows = [
    summarize_input_dataset("train_sessions", train_sessions_df, TRAIN_PARQUET_PATH),
    summarize_input_dataset("test_sessions", test_sessions_df, TEST_PARQUET_PATH),
    summarize_input_dataset("data_quality_issue_log", data_quality_issue_log_df, DATA_QUALITY_ISSUE_LOG_PATH),
]

input_overview_df = spark.createDataFrame(input_overview_rows)
input_overview_df.show(truncate=False)

print("Train schema:")
train_sessions_df.printSchema()

print("Test schema:")
test_sessions_df.printSchema()

print("Data quality issue log schema:")
data_quality_issue_log_df.printSchema()

+------------+----------------------+-------------------------------------------------+---------+---------------------+
|column_count|dataset               |path                                             |row_count|success_marker_exists|
+------------+----------------------+-------------------------------------------------+---------+---------------------+
|61          |train_sessions        |g:\ds\data_pyspark_parquet\train_sessions        |1708337  |true                 |
|61          |test_sessions         |g:\ds\data_pyspark_parquet\test_sessions         |401589   |true                 |
|9           |data_quality_issue_log|g:\ds\data_pyspark_parquet\data_quality_issue_log|94       |true                 |
+------------+----------------------+-------------------------------------------------+---------+---------------------+

Train schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitSta

#### Buoc 1.5: Xem sample va cac issue can uu tien

Xem mot vai dong dai dien cua train/test, dong thoi xem issue log theo severity va decision de cac buoc sau clean dung theo ket qua quality check.

In [5]:
preview_columns = [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "channelGrouping",
    "device_category",
    "geo_country",
    "traffic_medium_clean",
    "totals_transactions",
    "transaction_revenue",
    "total_transaction_revenue",
]

print("Train sample:")
train_sessions_df.select(preview_columns).show(5, truncate=False)

print("Test sample:")
test_sessions_df.select(preview_columns).show(5, truncate=False)

print("Issue count by severity and decision:")
(
    data_quality_issue_log_df
    .groupBy("severity", "decision")
    .count()
    .orderBy(F.desc("count"), "severity", "decision")
    .show(50, truncate=False)
)

print("High/medium issues to review first:")
(
    data_quality_issue_log_df
    .where(F.col("severity").isin("high", "medium"))
    .select(
        "issue_id",
        "dataset",
        "column",
        "issue_type",
        "affected_rows",
        "affected_pct",
        "severity",
        "decision",
        "reason",
    )
    .orderBy(F.desc("affected_pct"), "dataset", "column")
    .show(100, truncate=False)
)

Train sample:
+-------------------+----------+------------+---------------+---------------+-------------+--------------------+-------------------+-------------------+-------------------------+
|fullVisitorId      |visit_id  |session_date|channelGrouping|device_category|geo_country  |traffic_medium_clean|totals_transactions|transaction_revenue|total_transaction_revenue|
+-------------------+----------+------------+---------------+---------------+-------------+--------------------+-------------------+-------------------+-------------------------+
|0675225896207031578|1479521643|2016-11-18  |Organic Search |mobile         |Mexico       |(none)              |0                  |0.0                |0.0                      |
|0405493868621572321|1479500779|2016-11-18  |Organic Search |desktop        |Poland       |(none)              |0                  |0.0                |0.0                      |
|3943885042251597234|1479459642|2016-11-18  |Direct         |mobile         |Pakistan     |

### Buoc 2: Xu ly duplicate session

Muc tieu: dua du lieu ve dung grain session-level, moi session key chi con 1 dong.

Rule dedupe cho bai toan future 30 ngay:
1. Uu tien dong co raw revenue/transaction signal.
2. Uu tien revenue signal lon hon.
3. Uu tien `totals_transactions`, `totals_hits`, `totals_pageviews` lon hon.
4. Neu van trung thi uu tien session co thoi gian moi hon.
5. Neu van bang nhau thi dung hash cua toan dong de tie-break on dinh.

#### Buoc 2.1: Xem lai issue duplicate tu quality log

Loc cac issue co lien quan den duplicate/dedupe de xac nhan ly do can xu ly truoc EDA/model.

In [6]:
duplicate_issue_log_df = (
    data_quality_issue_log_df
    .where(
        F.lower(F.coalesce(F.col("issue_type"), F.lit(""))).contains("duplicate")
        | F.lower(F.coalesce(F.col("decision"), F.lit(""))).contains("dedupe")
    )
    .select(
        "issue_id",
        "dataset",
        "column",
        "issue_type",
        "affected_rows",
        "affected_pct",
        "severity",
        "decision",
        "reason",
    )
    .orderBy("dataset", "issue_id")
)

duplicate_issue_log_df.show(50, truncate=False)

+--------+--------------+------------------------+---------------------+-------------+------------+--------+-----------------------+----------------------------------------------------------------------------------------------+
|issue_id|dataset       |column                  |issue_type           |affected_rows|affected_pct|severity|decision               |reason                                                                                        |
+--------+--------------+------------------------+---------------------+-------------+------------+--------+-----------------------+----------------------------------------------------------------------------------------------+
|DQ-0043 |test_sessions |fullVisitorId + visit_id|duplicate_session_key|477          |0.1188      |medium  |dedupe_before_eda_model|Session-level key is not unique; duplicate sessions can bias timeline and user-level features.|
|DQ-0042 |train_sessions|fullVisitorId + visit_id|duplicate_session_key|1724         |0.

#### Buoc 2.2: Kiem tra duplicate truoc khi xu ly

Tinh lai duplicate tren du lieu dang doc de co baseline truoc khi dedupe.

In [7]:
SESSION_KEY_COLUMNS = ["fullVisitorId", "visit_id"]


def get_input_row_count(name, df):
    if "input_overview_rows" in globals():
        for row in input_overview_rows:
            if row["dataset"] == name:
                return row["row_count"]
    return df.count()


def build_duplicate_session_check(name, df, use_known_row_count=True):
    total_rows = get_input_row_count(name, df) if use_known_row_count else df.count()
    null_key_rows = df.where(
        F.col("fullVisitorId").isNull() | F.col("visit_id").isNull()
    ).count()

    key_counts_df = (
        df.groupBy(*SESSION_KEY_COLUMNS)
        .count()
        .persist(StorageLevel.DISK_ONLY)
    )

    distinct_session_keys = key_counts_df.count()
    duplicate_key_count = key_counts_df.where(F.col("count") > 1).count()

    duplicate_stats = (
        key_counts_df
        .where(F.col("count") > 1)
        .agg(
            F.coalesce(F.sum(F.col("count") - F.lit(1)), F.lit(0)).cast("long").alias("duplicate_extra_rows"),
            F.coalesce(F.max("count"), F.lit(0)).cast("long").alias("max_rows_per_key"),
        )
        .collect()[0]
    )

    key_counts_df.unpersist()

    duplicate_extra_rows = duplicate_stats["duplicate_extra_rows"]
    return {
        "dataset": name,
        "row_count": total_rows,
        "null_key_rows": null_key_rows,
        "distinct_session_keys": distinct_session_keys,
        "duplicate_key_count": duplicate_key_count,
        "duplicate_extra_rows": duplicate_extra_rows,
        "duplicate_extra_pct": round(duplicate_extra_rows / total_rows * 100, 4) if total_rows else 0.0,
        "max_rows_per_key": duplicate_stats["max_rows_per_key"],
    }


before_duplicate_check_rows = [
    build_duplicate_session_check("train_sessions", train_sessions_df),
    build_duplicate_session_check("test_sessions", test_sessions_df),
]

before_duplicate_check_df = spark.createDataFrame(before_duplicate_check_rows)
before_duplicate_check_df.show(truncate=False)

+--------------+---------------------+-------------------+--------------------+-------------------+----------------+-------------+---------+
|dataset       |distinct_session_keys|duplicate_extra_pct|duplicate_extra_rows|duplicate_key_count|max_rows_per_key|null_key_rows|row_count|
+--------------+---------------------+-------------------+--------------------+-------------------+----------------+-------------+---------+
|train_sessions|1706613              |0.1009             |1724                |1724               |2               |0            |1708337  |
|test_sessions |401112               |0.1188             |477                 |477                |2               |0            |401589   |
+--------------+---------------------+-------------------+--------------------+-------------------+----------------+-------------+---------+



#### Buoc 2.3: Xem sample duplicate key

Xem mot so duplicate session de biet cac dong trung co khac nhau ve raw revenue/transaction/hits/pageviews hay khong.

In [8]:
duplicate_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "visit_start_timestamp",
    "transaction_revenue",
    "total_transaction_revenue",
    "totals_transactions",
    "totals_hits",
    "totals_pageviews",
    "channelGrouping",
    "device_category",
    "traffic_medium_clean",
]


def show_duplicate_session_examples(name, df, limit=20):
    duplicate_keys_df = (
        df.groupBy(*SESSION_KEY_COLUMNS)
        .count()
        .where(F.col("count") > 1)
    )

    print(f"{name} duplicate examples:")
    (
        df.join(duplicate_keys_df.select(*SESSION_KEY_COLUMNS), on=SESSION_KEY_COLUMNS, how="inner")
        .select([column for column in duplicate_preview_columns if column in df.columns])
        .orderBy("fullVisitorId", "visit_id", F.desc("transaction_revenue"), F.desc("totals_transactions"))
        .show(limit, truncate=False)
    )


show_duplicate_session_examples("train_sessions", train_sessions_df)
show_duplicate_session_examples("test_sessions", test_sessions_df)

train_sessions duplicate examples:
+-------------------+----------+------------+---------------------+-------------------+-------------------------+-------------------+-----------+----------------+---------------+---------------+--------------------+
|fullVisitorId      |visit_id  |session_date|visit_start_timestamp|transaction_revenue|total_transaction_revenue|totals_transactions|totals_hits|totals_pageviews|channelGrouping|device_category|traffic_medium_clean|
+-------------------+----------+------------+---------------------+-------------------+-------------------------+-------------------+-----------+----------------+---------------+---------------+--------------------+
|0011338928267756760|1471848731|2016-08-22  |2016-08-22 07:02:10  |0.0                |0.0                      |0                  |2          |1               |Organic Search |desktop        |organic             |
|0011338928267756760|1471848731|2016-08-21  |2016-08-22 06:52:11  |0.0                |0.0           

#### Buoc 2.4: Dedupe theo rule uu tien

Dung window function de rank cac dong trong cung session key, sau do giu dong co rank = 1.

In [9]:
REVENUE_PRIORITY_COLUMNS = [
    "transaction_revenue",
    "total_transaction_revenue",
    "totals_transaction_revenue",
    "totals_total_transaction_revenue",
]
REVENUE_MICRO_DIVISOR = 1_000_000.0


def revenue_priority_expr(df):
    expressions = []
    for column in REVENUE_PRIORITY_COLUMNS:
        if column not in df.columns:
            continue
        value = F.coalesce(F.col(column).cast("double"), F.lit(0.0))
        if column.startswith("totals_"):
            value = value / F.lit(REVENUE_MICRO_DIVISOR)
        expressions.append(value)
    return F.greatest(*expressions) if expressions else F.lit(0.0)


def numeric_priority_column(df, column_name, cast_type="double", default_value=0):
    if column_name not in df.columns:
        return F.lit(default_value)
    return F.coalesce(F.col(column_name).cast(cast_type), F.lit(default_value))


def add_row_hash(df):
    hash_columns = [
        F.coalesce(F.col(column).cast("string"), F.lit("<null>"))
        for column in sorted(df.columns)
    ]
    return F.sha2(F.concat_ws("||", *hash_columns), 256)


def dedupe_session_df(df):
    dedupe_revenue = revenue_priority_expr(df)
    dedupe_transaction_count = numeric_priority_column(df, "totals_transactions", "long", 0)
    purchase_signal_priority = F.when((dedupe_revenue > 0) | (dedupe_transaction_count > 0), F.lit(1)).otherwise(F.lit(0))
    dedupe_time = F.coalesce(
        numeric_priority_column(df, "visit_start_time", "long", 0),
        numeric_priority_column(df, "visit_id", "long", 0),
    )

    ranked_df = (
        df
        .withColumn("_dedupe_purchase_signal_priority", purchase_signal_priority)
        .withColumn("_dedupe_revenue", dedupe_revenue)
        .withColumn("_dedupe_transaction_count", dedupe_transaction_count)
        .withColumn("_dedupe_hits", numeric_priority_column(df, "totals_hits", "long", 0))
        .withColumn("_dedupe_pageviews", numeric_priority_column(df, "totals_pageviews", "long", 0))
        .withColumn("_dedupe_time", dedupe_time)
        .withColumn("_dedupe_row_hash", add_row_hash(df))
    )

    dedupe_window = Window.partitionBy(*SESSION_KEY_COLUMNS).orderBy(
        F.desc("_dedupe_purchase_signal_priority"),
        F.desc("_dedupe_revenue"),
        F.desc("_dedupe_transaction_count"),
        F.desc("_dedupe_hits"),
        F.desc("_dedupe_pageviews"),
        F.desc("_dedupe_time"),
        F.asc("_dedupe_row_hash"),
    )

    return (
        ranked_df
        .withColumn("_dedupe_rank", F.row_number().over(dedupe_window))
        .where(F.col("_dedupe_rank") == 1)
        .drop(
            "_dedupe_purchase_signal_priority",
            "_dedupe_revenue",
            "_dedupe_transaction_count",
            "_dedupe_hits",
            "_dedupe_pageviews",
            "_dedupe_time",
            "_dedupe_row_hash",
            "_dedupe_rank",
        )
    )


train_sessions_dedup_df = dedupe_session_df(train_sessions_df).persist(StorageLevel.DISK_ONLY)
test_sessions_dedup_df = dedupe_session_df(test_sessions_df).persist(StorageLevel.DISK_ONLY)

print("Da tao train_sessions_dedup_df va test_sessions_dedup_df.")

Da tao train_sessions_dedup_df va test_sessions_dedup_df.


#### Buoc 2.5: Validation sau dedupe

Kiem tra lai row count va duplicate session key sau khi dedupe. Ket qua mong muon: `duplicate_extra_rows = 0`.

In [10]:
after_duplicate_check_rows = [
    build_duplicate_session_check("train_sessions", train_sessions_dedup_df, use_known_row_count=False),
    build_duplicate_session_check("test_sessions", test_sessions_dedup_df, use_known_row_count=False),
]

after_duplicate_check_df = spark.createDataFrame(after_duplicate_check_rows)

before_by_dataset = {row["dataset"]: row for row in before_duplicate_check_rows}
after_by_dataset = {row["dataset"]: row for row in after_duplicate_check_rows}

dedupe_validation_rows = []
for dataset, before_row in before_by_dataset.items():
    after_row = after_by_dataset[dataset]
    removed_duplicate_rows = before_row["row_count"] - after_row["row_count"]
    dedupe_validation_rows.append({
        "dataset": dataset,
        "before_rows": before_row["row_count"],
        "after_rows": after_row["row_count"],
        "removed_duplicate_rows": removed_duplicate_rows,
        "expected_removed_duplicate_rows": before_row["duplicate_extra_rows"],
        "duplicate_extra_rows_after": after_row["duplicate_extra_rows"],
        "duplicate_key_count_after": after_row["duplicate_key_count"],
        "null_key_rows_after": after_row["null_key_rows"],
    })

dedupe_validation_df = spark.createDataFrame(dedupe_validation_rows)
dedupe_validation_df.show(truncate=False)

for row in dedupe_validation_rows:
    if row["duplicate_extra_rows_after"] != 0:
        raise AssertionError(f"Duplicate session key van con sau dedupe: {row}")

print("Validation dedupe thanh cong: duplicate session key sau clean = 0.")

+----------+-----------+--------------+--------------------------+-------------------------+-------------------------------+-------------------+----------------------+
|after_rows|before_rows|dataset       |duplicate_extra_rows_after|duplicate_key_count_after|expected_removed_duplicate_rows|null_key_rows_after|removed_duplicate_rows|
+----------+-----------+--------------+--------------------------+-------------------------+-------------------------------+-------------------+----------------------+
|1706613   |1708337    |train_sessions|0                         |0                        |1724                           |0                  |1724                  |
|401112    |401589     |test_sessions |0                         |0                        |477                            |0                  |477                   |
+----------+-----------+--------------+--------------------------+-------------------------+-------------------------------+-------------------+----------------

### Buoc 3: Chuan hoa placeholder va boolean null

Muc tieu: thay cac gia tri placeholder tho thanh gia tri chuan de EDA/model khong bi nhieu boi nhieu cach bieu dien missing.

Nguyen tac xu ly trong buoc nay:
- Cot categorical dung cho EDA/model: doi placeholder thanh `unknown`.
- Cot boolean: fill null thanh `False` neu logic hop ly.
- Cot id/time/numeric/target revenue: khong xu ly placeholder trong buoc nay vi can giu dung kieu du lieu.
- Cot qua thieu nhieu chua drop o day; viec drop/giu/giu flag se lam o buoc tiep theo.

Viec can lam:
1. Khai bao danh sach placeholder da xac nhan tu notebook 02.
2. Chon cac cot categorical can chuan hoa.
3. Kiem tra placeholder truoc khi xu ly.
4. Tao DataFrame da chuan hoa placeholder.
5. Validation de dam bao khong con placeholder tho trong cac cot quan trong.

#### Buoc 3.1: Khai bao rule placeholder

Dung lai cac placeholder da xac nhan tu quality check. Gia tri `unknown` duoc xem la gia tri chuan sau clean, con cac placeholder tho khac se duoc doi ve `unknown`.

In [11]:
STANDARD_UNKNOWN_VALUE = "unknown"

CONFIRMED_PLACEHOLDER_VALUES = [
    "(not set)",
    "not available in demo dataset",
    "unknown.unknown",
    "(not provided)",
    "not provided",
    "unknown",
    "",
]

# Sau clean, `unknown` la gia tri chuan nen khong con bi xem la placeholder tho.
NON_STANDARD_PLACEHOLDER_VALUES = [
    value for value in CONFIRMED_PLACEHOLDER_VALUES
    if value != STANDARD_UNKNOWN_VALUE
]

CATEGORICAL_PLACEHOLDER_COLUMNS = [
    "channelGrouping",
    "socialEngagementType",
    "device_browser",
    "device_operating_system",
    "device_category",
    "browser_family",
    "os_family",
    "geo_continent",
    "geo_sub_continent",
    "geo_country",
    "geo_region",
    "geo_metro",
    "geo_city",
    "geo_network_domain",
    "geo_region_clean",
    "traffic_campaign",
    "traffic_source",
    "traffic_medium",
    "traffic_keyword",
    "traffic_referral_path",
    "traffic_ad_content",
    "traffic_gcl_id",
    "traffic_medium_clean",
    "traffic_source_clean",
    "traffic_channel_type",
    "custom_dimension_value",
    "purchase_signal_reason",
]

BOOLEAN_FILL_FALSE_COLUMNS = [
    "device_is_mobile",
    "traffic_is_true_direct",
]


def existing_columns(df, columns):
    return [column for column in columns if column in df.columns]


categorical_placeholder_columns = existing_columns(train_sessions_dedup_df, CATEGORICAL_PLACEHOLDER_COLUMNS)
boolean_fill_false_columns = existing_columns(train_sessions_dedup_df, BOOLEAN_FILL_FALSE_COLUMNS)

print("Categorical columns to normalize:", categorical_placeholder_columns)
print("Boolean columns to fill False:", boolean_fill_false_columns)

Categorical columns to normalize: ['channelGrouping', 'socialEngagementType', 'device_browser', 'device_operating_system', 'device_category', 'browser_family', 'os_family', 'geo_continent', 'geo_sub_continent', 'geo_country', 'geo_region', 'geo_metro', 'geo_city', 'geo_network_domain', 'geo_region_clean', 'traffic_campaign', 'traffic_source', 'traffic_medium', 'traffic_keyword', 'traffic_referral_path', 'traffic_ad_content', 'traffic_gcl_id', 'traffic_medium_clean', 'traffic_source_clean', 'traffic_channel_type', 'custom_dimension_value']
Boolean columns to fill False: ['device_is_mobile', 'traffic_is_true_direct']


#### Buoc 3.2: Kiem tra placeholder truoc khi xu ly

Tinh null, placeholder tho va `unknown` hien co trong cac cot categorical de co baseline truoc khi chuan hoa.

In [12]:
def normalized_string_expr(column_name):
    return F.lower(F.trim(F.coalesce(F.col(column_name).cast("string"), F.lit(""))))


def build_placeholder_summary(name, df, columns, placeholder_values):
    total_rows = df.count()
    rows = []

    for column_name in columns:
        normalized_value = normalized_string_expr(column_name)
        stats = df.agg(
            F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias("null_count"),
            F.sum(F.when(normalized_value.isin(placeholder_values), 1).otherwise(0)).alias("placeholder_count"),
            F.sum(F.when(normalized_value == STANDARD_UNKNOWN_VALUE, 1).otherwise(0)).alias("unknown_count"),
        ).collect()[0]

        rows.append({
            "dataset": name,
            "column": column_name,
            "row_count": total_rows,
            "null_count": int(stats["null_count"] or 0),
            "placeholder_count": int(stats["placeholder_count"] or 0),
            "unknown_count": int(stats["unknown_count"] or 0),
            "null_pct": round((stats["null_count"] or 0) / total_rows * 100, 4) if total_rows else 0.0,
            "placeholder_pct": round((stats["placeholder_count"] or 0) / total_rows * 100, 4) if total_rows else 0.0,
            "unknown_pct": round((stats["unknown_count"] or 0) / total_rows * 100, 4) if total_rows else 0.0,
        })

    return rows


before_placeholder_summary_rows = (
    build_placeholder_summary("train_sessions", train_sessions_dedup_df, categorical_placeholder_columns, CONFIRMED_PLACEHOLDER_VALUES)
    + build_placeholder_summary("test_sessions", test_sessions_dedup_df, categorical_placeholder_columns, CONFIRMED_PLACEHOLDER_VALUES)
)

before_placeholder_summary_df = spark.createDataFrame(before_placeholder_summary_rows)
(
    before_placeholder_summary_df
    .where((F.col("null_count") > 0) | (F.col("placeholder_count") > 0))
    .orderBy(F.desc("placeholder_pct"), F.desc("null_pct"), "dataset", "column")
    .show(100, truncate=False)
)

+-----------------------+--------------+----------+--------+-----------------+---------------+---------+-------------+-----------+
|column                 |dataset       |null_count|null_pct|placeholder_count|placeholder_pct|row_count|unknown_count|unknown_pct|
+-----------------------+--------------+----------+--------+-----------------+---------------+---------+-------------+-----------+
|traffic_gcl_id         |test_sessions |390508    |97.3563 |390508           |97.3563        |401112   |0            |0.0        |
|traffic_ad_content     |test_sessions |0         |0.0     |390373           |97.3227        |401112   |0            |0.0        |
|traffic_keyword        |test_sessions |40192     |10.0201 |389062           |96.9959        |401112   |0            |0.0        |
|traffic_ad_content     |train_sessions|1641904   |96.2083 |1641904          |96.2083        |1706613  |0            |0.0        |
|traffic_gcl_id         |train_sessions|1631247   |95.5839 |1631247          |95.58

#### Buoc 3.3: Ap dung chuan hoa placeholder

Voi categorical, trim khoang trang va dua cac placeholder ve `unknown`. Voi boolean, fill null thanh `False`.

In [13]:
def clean_categorical_placeholder(column_name):
    trimmed_value = F.trim(F.col(column_name).cast("string"))
    normalized_value = F.lower(F.coalesce(trimmed_value, F.lit("")))

    return (
        F.when(F.col(column_name).isNull(), F.lit(STANDARD_UNKNOWN_VALUE))
        .when(normalized_value.isin(CONFIRMED_PLACEHOLDER_VALUES), F.lit(STANDARD_UNKNOWN_VALUE))
        .otherwise(trimmed_value)
    )


def normalize_placeholders_df(df):
    cleaned_df = df

    for column_name in existing_columns(df, CATEGORICAL_PLACEHOLDER_COLUMNS):
        cleaned_df = cleaned_df.withColumn(column_name, clean_categorical_placeholder(column_name))

    for column_name in existing_columns(df, BOOLEAN_FILL_FALSE_COLUMNS):
        cleaned_df = cleaned_df.withColumn(column_name, F.coalesce(F.col(column_name).cast("boolean"), F.lit(False)))

    return cleaned_df


train_sessions_placeholder_clean_df = normalize_placeholders_df(train_sessions_dedup_df).persist(StorageLevel.DISK_ONLY)
test_sessions_placeholder_clean_df = normalize_placeholders_df(test_sessions_dedup_df).persist(StorageLevel.DISK_ONLY)

print("Da tao train_sessions_placeholder_clean_df va test_sessions_placeholder_clean_df.")

Da tao train_sessions_placeholder_clean_df va test_sessions_placeholder_clean_df.


#### Buoc 3.4: Validation sau chuan hoa placeholder

Kiem tra cac placeholder tho khong con trong cot categorical quan trong. Gia tri `unknown` duoc giu lai nhu missing bucket chuan.

In [14]:
after_placeholder_summary_rows = (
    build_placeholder_summary("train_sessions", train_sessions_placeholder_clean_df, categorical_placeholder_columns, NON_STANDARD_PLACEHOLDER_VALUES)
    + build_placeholder_summary("test_sessions", test_sessions_placeholder_clean_df, categorical_placeholder_columns, NON_STANDARD_PLACEHOLDER_VALUES)
)

after_placeholder_summary_df = spark.createDataFrame(after_placeholder_summary_rows)

placeholder_validation_df = (
    after_placeholder_summary_df
    .where((F.col("null_count") > 0) | (F.col("placeholder_count") > 0))
    .orderBy(F.desc("placeholder_pct"), F.desc("null_pct"), "dataset", "column")
)

placeholder_validation_df.show(100, truncate=False)

remaining_placeholder_rows = placeholder_validation_df.count()
if remaining_placeholder_rows > 0:
    raise AssertionError("Van con null hoac placeholder tho trong categorical columns sau clean.")


def build_boolean_null_summary(name, df, columns):
    total_rows = df.count()
    rows = []
    for column_name in columns:
        null_count = df.where(F.col(column_name).isNull()).count()
        rows.append({
            "dataset": name,
            "column": column_name,
            "row_count": total_rows,
            "null_count": null_count,
            "null_pct": round(null_count / total_rows * 100, 4) if total_rows else 0.0,
        })
    return rows


boolean_null_summary_rows = (
    build_boolean_null_summary("train_sessions", train_sessions_placeholder_clean_df, boolean_fill_false_columns)
    + build_boolean_null_summary("test_sessions", test_sessions_placeholder_clean_df, boolean_fill_false_columns)
)

boolean_null_summary_df = spark.createDataFrame(boolean_null_summary_rows)
boolean_null_summary_df.show(truncate=False)

for row in boolean_null_summary_rows:
    if row["null_count"] != 0:
        raise AssertionError(f"Boolean column van con null sau fill: {row}")

print("Validation placeholder thanh cong: categorical khong con placeholder tho, boolean khong con null.")

+------+-------+----------+--------+-----------------+---------------+---------+-------------+-----------+
|column|dataset|null_count|null_pct|placeholder_count|placeholder_pct|row_count|unknown_count|unknown_pct|
+------+-------+----------+--------+-----------------+---------------+---------+-------------+-----------+
+------+-------+----------+--------+-----------------+---------------+---------+-------------+-----------+

+----------------------+--------------+----------+--------+---------+
|column                |dataset       |null_count|null_pct|row_count|
+----------------------+--------------+----------+--------+---------+
|device_is_mobile      |train_sessions|0         |0.0     |1706613  |
|traffic_is_true_direct|train_sessions|0         |0.0     |1706613  |
|device_is_mobile      |test_sessions |0         |0.0     |401112   |
|traffic_is_true_direct|test_sessions |0         |0.0     |401112   |
+----------------------+--------------+----------+--------+---------+

Validation

### Buoc 4: Phan tich cot can giu, bo va sua doi

Muc tieu: tao mot bang quyet dinh ro rang cho tung nhom cot truoc khi di vao EDA/model.

Trong buoc nay can phan biet 2 viec:
- Giu trong clean dataset: cot van co ich cho EDA, trace, validation hoac API.
- Dung lam feature model: cot duoc phep dua vao encoding/model sau nay.

Viec can lam:
1. Phan nhom cot theo vai tro: key, time, categorical, numeric, sparse, target/leakage.
2. Ghi decision cho tung cot: `keep_as_feature`, `keep_key`, `keep_for_eda`, `keep_flag_only`, `drop_from_model`, `target_leakage`, `review_before_model`.
3. Mo ta cot nao can sua doi va sua nhu the nao.
4. Tao cac flag thay the cho cot sparse/high-missing.
5. Tao danh sach cot exclude khoi model va danh sach feature candidate cho cac buoc sau.

#### Buoc 4.1: Nguyen tac quyet dinh cot

| Nhom cot | Quyet dinh | Ly do |
|---|---|---|
| Session/user key | Giu trong dataset, khong dung truc tiep lam feature | Can cho dedupe, aggregate user-level va API |
| Raw id/time string | Giu de trace, drop khoi model | Da co ban cast sach nhu `visit_id`, `visit_number`, `session_date` |
| Revenue/transaction source signal | Giu de tao future label, cam dung lam feature | Day la tin hieu mua hang cua session hien tai, co the gay leakage neu dua vao model |
| Sparse/high-missing categorical | Khong dung raw cho model, tao flag | Raw value qua thieu/high-cardinality |
| Clean categorical chinh | Giu lam feature candidate | Can lower/trim va gom rare category o buoc sau |
| Numeric hanh vi session | Giu lam feature candidate neu khong leakage | Phuc vu EDA/model session va aggregate user-level |

#### Buoc 4.2: Khai bao bang column treatment plan

Bang nay la decision table trung tam cho notebook clean. Moi cot co:
- `role`: vai tro cua cot.
- `decision`: cot duoc giu/bo/dung the nao.
- `model_usage`: co duoc dung lam feature model hay khong.
- `clean_action`: can sua doi nhu the nao.
- `reason`: ly do quyet dinh.

In [15]:
COLUMN_TREATMENT_OVERRIDES = {
    # Key va raw id/time.
    "fullVisitorId": {
        "role": "user_key",
        "decision": "keep_key",
        "model_usage": "exclude_direct_feature",
        "clean_action": "keep_for_user_aggregate_and_api",
        "reason": "User id can cho aggregate user-level va API, nhung cardinality qua cao de lam feature truc tiep.",
    },
    "visit_id": {
        "role": "session_key",
        "decision": "keep_key",
        "model_usage": "exclude_direct_feature",
        "clean_action": "keep_for_session_trace",
        "reason": "Session id dung de dedupe/trace, khong nen dua vao model.",
    },
    "visitId": {
        "role": "raw_session_key",
        "decision": "drop_from_model",
        "model_usage": "exclude",
        "clean_action": "keep_for_trace_only_use_visit_id_instead",
        "reason": "Ban raw string trung thong tin voi visit_id da cast.",
    },
    "visitNumber": {
        "role": "raw_numeric_string",
        "decision": "drop_from_model",
        "model_usage": "exclude",
        "clean_action": "use_visit_number_instead",
        "reason": "Ban raw string, da co visit_number kieu int.",
    },
    "visitStartTime": {
        "role": "raw_time_string",
        "decision": "drop_from_model",
        "model_usage": "exclude",
        "clean_action": "use_visit_start_time_or_visit_start_timestamp_instead",
        "reason": "Ban raw string, da co timestamp/time sach hon.",
    },
    "date": {
        "role": "raw_date_string",
        "decision": "drop_from_model",
        "model_usage": "exclude",
        "clean_action": "use_session_date_and_time_parts_instead",
        "reason": "Ban raw yyyyMMdd, da co session_date va year/month/day.",
    },

    # Time clean.
    "visit_number": {
        "role": "numeric_behavior",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_non_negative_integer",
        "reason": "So lan visit la tin hieu hanh vi co ich.",
    },
    "visit_start_time": {
        "role": "time_trace",
        "decision": "keep_for_eda",
        "model_usage": "exclude_direct_feature",
        "clean_action": "derive_time_parts_for_model",
        "reason": "Unix timestamp raw co the gay leakage theo split thoi gian; dung time parts thay the.",
    },
    "visit_start_timestamp": {
        "role": "time_trace",
        "decision": "keep_for_eda",
        "model_usage": "exclude_direct_feature",
        "clean_action": "use_for_eda_and_validation",
        "reason": "Can cho EDA/time validation, khong dung truc tiep lam feature.",
    },
    "session_date": {
        "role": "time_trace",
        "decision": "keep_for_eda",
        "model_usage": "exclude_direct_feature",
        "clean_action": "use_for_partition_eda_and_split",
        "reason": "Can cho EDA va split theo thoi gian, khong dua date raw vao model.",
    },
    "session_year": {
        "role": "time_part",
        "decision": "keep_for_eda",
        "model_usage": "review_before_model",
        "clean_action": "keep_for_partition_and_time_shift_check",
        "reason": "Co ich cho partition/EDA, nhung co the hoc theo split thoi gian.",
    },
    "session_month": {
        "role": "time_part",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_integer_1_12",
        "reason": "Thang co the bat seasonality.",
    },
    "session_day_of_week": {
        "role": "time_part",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_integer_1_7",
        "reason": "Ngay trong tuan co the lien quan hanh vi truy cap/mua hang.",
    },
    "session_hour": {
        "role": "time_part",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_integer_0_23",
        "reason": "Gio truy cap la feature hanh vi co the dung.",
    },

    # Target/leakage.
    "purchase_signal_reason": {
        "role": "target_debug",
        "decision": "target_leakage",
        "model_usage": "exclude",
        "clean_action": "keep_for_label_debug_only",
        "reason": "Giai thich label, tiet lo purchase signal.",
    },
    "transaction_revenue": {
        "role": "label_source_signal",
        "decision": "target_leakage",
        "model_usage": "target_only",
        "clean_action": "use_to_create_session_revenue_signal",
        "reason": "Revenue source signal cua session hien tai, dung de tao future label; khong duoc lam feature input.",
    },
    "total_transaction_revenue": {
        "role": "label_source_signal",
        "decision": "target_leakage",
        "model_usage": "target_only",
        "clean_action": "compare_source_revenue_then_exclude_from_features",
        "reason": "Revenue source/debug cua session hien tai, co leakage neu dung lam feature.",
    },
    "totals_transaction_revenue": {
        "role": "label_source_signal_raw_micro",
        "decision": "target_leakage",
        "model_usage": "exclude",
        "clean_action": "keep_for_revenue_validation_only",
        "reason": "Revenue micros la source signal cua session hien tai, exclude khoi feature.",
    },
    "totals_total_transaction_revenue": {
        "role": "label_source_signal_raw_micro",
        "decision": "target_leakage",
        "model_usage": "exclude",
        "clean_action": "keep_for_revenue_validation_only",
        "reason": "Revenue micros la source signal cua session hien tai, exclude khoi feature.",
    },
    "totals_transactions": {
        "role": "target_leakage_signal",
        "decision": "target_leakage",
        "model_usage": "exclude",
        "clean_action": "keep_for_label_validation_only",
        "reason": "So transaction tiet lo truc tiep viec mua hang.",
    },

    # Device/browser.
    "device_browser": {
        "role": "raw_high_cardinality_category",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw_use_family",
        "clean_action": "use_browser_family_for_model",
        "reason": "Browser raw co nhieu gia tri, family on dinh hon cho model.",
    },
    "device_operating_system": {
        "role": "raw_high_cardinality_category",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw_use_family",
        "clean_action": "use_os_family_for_model",
        "reason": "OS raw co nhieu bien the, os_family on dinh hon.",
    },
    "browser_family": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Browser family it nhieu hon raw browser va co y nghia hanh vi.",
    },
    "os_family": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "OS family la categorical feature gon hon raw OS.",
    },
    "device_category": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim",
        "reason": "Mobile/desktop/tablet co y nghia ro cho hanh vi truy cap.",
    },
    "device_is_mobile": {
        "role": "boolean_feature",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "fill_null_false_done",
        "reason": "Boolean device mobile de model xu ly truc tiep.",
    },

    # Geo.
    "geo_continent": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Geo level rong, cardinality thap hon country/city.",
    },
    "geo_sub_continent": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Co y nghia dia ly va it granular hon city.",
    },
    "geo_country": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Country quan trong cho EDA/model, can gom rare category.",
    },
    "geo_region": {
        "role": "sparse_geo_category",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw_use_geo_region_clean_or_flag",
        "clean_action": "use_geo_region_clean_and_has_geo_region",
        "reason": "Region raw co placeholder/sparse; dung ban clean va flag tot hon.",
    },
    "geo_region_clean": {
        "role": "clean_categorical",
        "decision": "review_before_model",
        "model_usage": "review_before_model",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Region co the huu ich nhung cardinality/sparsity can xem lai truoc model.",
    },
    "geo_city": {
        "role": "high_cardinality_geo",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw",
        "clean_action": "use_country_or_region_instead",
        "reason": "City qua high-cardinality, de gay noise khi encoding.",
    },
    "geo_metro": {
        "role": "sparse_geo_category",
        "decision": "keep_flag_only",
        "model_usage": "use_flag_not_raw",
        "clean_action": "create_has_geo_metro_then_exclude_raw_from_model",
        "reason": "Metro thieu nhieu, raw category khong on dinh.",
    },
    "geo_network_domain": {
        "role": "high_cardinality_geo",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw",
        "clean_action": "exclude_from_model",
        "reason": "Network domain high-cardinality va co nhieu gia tri placeholder.",
    },
    "has_geo_region": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Flag availability an toan hon raw missing region.",
    },

    # Traffic.
    "channelGrouping": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Kenh truy cap la feature business quan trong.",
    },
    "socialEngagementType": {
        "role": "low_variance_category",
        "decision": "keep_flag_only",
        "model_usage": "use_flag_not_raw",
        "clean_action": "create_is_socially_engaged_then_exclude_raw_from_model",
        "reason": "Raw value thuong it bien thien; flag de dung hon.",
    },
    "traffic_source": {
        "role": "raw_traffic_category",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw_use_clean",
        "clean_action": "use_traffic_source_clean_for_model",
        "reason": "Da co ban clean rieng de encode on dinh hon.",
    },
    "traffic_medium": {
        "role": "raw_traffic_category",
        "decision": "keep_for_eda",
        "model_usage": "exclude_raw_use_clean",
        "clean_action": "use_traffic_medium_clean_for_model",
        "reason": "Da co traffic_medium_clean.",
    },
    "traffic_source_clean": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim_and_group_rare_later",
        "reason": "Traffic source co y nghia cao, can gom rare category.",
    },
    "traffic_medium_clean": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim",
        "reason": "Medium la feature traffic quan trong.",
    },
    "traffic_channel_type": {
        "role": "clean_categorical",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "lower_trim",
        "reason": "Nhom channel da duoc chuan hoa san.",
    },
    "traffic_campaign": {
        "role": "sparse_traffic_category",
        "decision": "keep_flag_only",
        "model_usage": "use_flag_not_raw",
        "clean_action": "create_has_traffic_campaign_then_exclude_raw_from_model",
        "reason": "Campaign thieu nhieu va high-cardinality.",
    },
    "traffic_keyword": {
        "role": "sparse_traffic_category",
        "decision": "keep_flag_only",
        "model_usage": "use_existing_has_traffic_keyword_not_raw",
        "clean_action": "exclude_raw_from_model",
        "reason": "Keyword thieu nhieu; da co has_traffic_keyword.",
    },
    "traffic_ad_content": {
        "role": "sparse_traffic_category",
        "decision": "keep_flag_only",
        "model_usage": "use_flag_not_raw",
        "clean_action": "create_has_traffic_ad_content_then_exclude_raw_from_model",
        "reason": "Ad content thieu nhieu/high-cardinality.",
    },
    "traffic_gcl_id": {
        "role": "sparse_traffic_identifier",
        "decision": "keep_flag_only",
        "model_usage": "use_existing_has_gclid_not_raw",
        "clean_action": "exclude_raw_from_model",
        "reason": "GCLID la ad id, high-cardinality va chi nen dung flag.",
    },
    "traffic_referral_path": {
        "role": "sparse_traffic_path",
        "decision": "keep_flag_only",
        "model_usage": "use_flag_not_raw",
        "clean_action": "create_has_referral_path_then_exclude_raw_from_model",
        "reason": "Referral path high-cardinality va sparse.",
    },
    "traffic_is_true_direct": {
        "role": "boolean_feature",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "fill_null_false_done",
        "reason": "Direct traffic boolean co y nghia model.",
    },
    "is_direct_traffic": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Traffic flag da clean.",
    },
    "is_paid_traffic": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Traffic flag da clean.",
    },
    "is_organic_traffic": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Traffic flag da clean.",
    },
    "is_referral_traffic": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Traffic flag da clean.",
    },
    "has_traffic_keyword": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Flag thay the raw keyword sparse.",
    },
    "has_gclid": {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1",
        "reason": "Flag paid ad click an toan hon raw gclid.",
    },

    # Custom dimension.
    "custom_dimension_value": {
        "role": "sparse_custom_category",
        "decision": "keep_for_eda",
        "model_usage": "review_before_model",
        "clean_action": "create_has_custom_dimension_and_review_values",
        "reason": "Can inspect them truoc khi dua vao model.",
    },

    # Numeric session behavior.
    "totals_visits": {
        "role": "low_information_numeric",
        "decision": "drop_from_model",
        "model_usage": "exclude",
        "clean_action": "keep_for_validation_only",
        "reason": "Thuong gan nhu constant = 1 o session-level.",
    },
    "totals_hits": {
        "role": "numeric_behavior",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_non_negative_integer",
        "reason": "So hits la tin hieu engagement.",
    },
    "totals_pageviews": {
        "role": "numeric_behavior",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_non_negative_integer",
        "reason": "Pageviews la tin hieu engagement.",
    },
    "totals_bounces": {
        "role": "numeric_behavior",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "also_create_is_bounce_later",
        "reason": "Bounce la tin hieu hanh vi; buoc sau tao is_bounce de de dung.",
    },
    "totals_new_visits": {
        "role": "numeric_behavior",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_0_1_or_count",
        "reason": "New visit co y nghia user behavior.",
    },
    "totals_time_on_site": {
        "role": "numeric_behavior",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "keep_non_negative_numeric",
        "reason": "Time on site la tin hieu engagement.",
    },
    "totals_session_quality_dim": {
        "role": "quality_score",
        "decision": "review_before_model",
        "model_usage": "review_before_model",
        "clean_action": "keep_for_eda_check_possible_leakage",
        "reason": "Co the la diem chat luong do GA tao, can review leakage truoc model.",
    },
}


DEFAULT_TREATMENT = {
    "role": "unclassified",
    "decision": "review_before_model",
    "model_usage": "review_before_model",
    "clean_action": "inspect_before_use",
    "reason": "Cot chua co rule rieng, can review truoc khi dua vao model.",
}


def build_column_treatment_plan(df):
    rows = []
    dtype_by_column = dict(df.dtypes)

    for column_name in df.columns:
        treatment = {**DEFAULT_TREATMENT, **COLUMN_TREATMENT_OVERRIDES.get(column_name, {})}
        rows.append({
            "column": column_name,
            "dtype": dtype_by_column[column_name],
            **treatment,
        })

    return spark.createDataFrame(rows)


column_treatment_plan_df = build_column_treatment_plan(train_sessions_placeholder_clean_df)

column_treatment_plan_df.orderBy("decision", "role", "column").show(200, truncate=False)

+---------------------------------------------------------+--------------------------------+-------------------+---------+----------------------------------------+---------------------------------------------------------------------------------------------------+-----------------------------+
|clean_action                                             |column                          |decision           |dtype    |model_usage                             |reason                                                                                             |role                         |
+---------------------------------------------------------+--------------------------------+-------------------+---------+----------------------------------------+---------------------------------------------------------------------------------------------------+-----------------------------+
|keep_for_validation_only                                 |totals_visits                   |drop_from_model    |int   

#### Buoc 4.3: Tom tat so cot theo decision

Bang nay giup nhin nhanh co bao nhieu cot duoc giu lam feature, bao nhieu cot chi giu de EDA/trace, va bao nhieu cot bi cam dua vao model vi leakage.

In [16]:
(
    column_treatment_plan_df
    .groupBy("decision", "model_usage")
    .count()
    .orderBy(F.desc("count"), "decision", "model_usage")
    .show(truncate=False)
)

print("Cot can loai khoi model hoac chi dung de target/debug:")
(
    column_treatment_plan_df
    .where(F.col("model_usage").isin(
        "exclude",
        "exclude_direct_feature",
        "exclude_raw",
        "exclude_raw_use_clean",
        "exclude_raw_use_family",
        "target_only",
    ) | F.col("decision").isin("target_leakage", "drop_from_model"))
    .select("column", "role", "decision", "model_usage", "clean_action", "reason")
    .orderBy("decision", "column")
    .show(200, truncate=False)
)

print("Cot can sua doi/tao flag truoc khi model:")
(
    column_treatment_plan_df
    .where(
        F.col("decision").isin("keep_flag_only", "review_before_model")
        | F.col("clean_action").contains("group_rare")
        | F.col("clean_action").contains("create_")
    )
    .select("column", "role", "decision", "model_usage", "clean_action", "reason")
    .orderBy("decision", "column")
    .show(200, truncate=False)
)

+-------------------+----------------------------------------+-----+
|decision           |model_usage                             |count|
+-------------------+----------------------------------------+-----+
|keep_as_feature    |candidate_feature                       |28   |
|drop_from_model    |exclude                                 |5    |
|keep_flag_only     |use_flag_not_raw                        |5    |
|keep_for_eda       |exclude_direct_feature                  |3    |
|target_leakage     |exclude                                 |3    |
|keep_for_eda       |exclude_raw                             |2    |
|keep_for_eda       |exclude_raw_use_clean                   |2    |
|keep_for_eda       |exclude_raw_use_family                  |2    |
|keep_for_eda       |review_before_model                     |2    |
|keep_key           |exclude_direct_feature                  |2    |
|review_before_model|review_before_model                     |2    |
|target_leakage     |target_only  

#### Buoc 4.4: Tao flag thay the cho cot sparse

Voi cac cot qua thieu hoac high-cardinality, khong dua raw value vao model. Thay vao do tao flag `has_*` de giu lai tin hieu co/khong co thong tin.

In [17]:
SPARSE_FLAG_SPECS = {
    "traffic_campaign": "has_traffic_campaign",
    "traffic_ad_content": "has_traffic_ad_content",
    "traffic_referral_path": "has_referral_path",
    "geo_metro": "has_geo_metro",
    "custom_dimension_value": "has_custom_dimension",
}


def has_non_unknown_value(column_name):
    normalized_value = F.lower(F.trim(F.coalesce(F.col(column_name).cast("string"), F.lit(""))))
    return F.when(
        normalized_value.isin(CONFIRMED_PLACEHOLDER_VALUES) | (normalized_value == STANDARD_UNKNOWN_VALUE),
        F.lit(0),
    ).otherwise(F.lit(1))


def add_sparse_indicator_flags(df):
    updated_df = df

    for source_column, flag_column in SPARSE_FLAG_SPECS.items():
        if source_column in updated_df.columns:
            updated_df = updated_df.withColumn(flag_column, has_non_unknown_value(source_column).cast("int"))

    if "socialEngagementType" in updated_df.columns:
        normalized_social = F.lower(F.trim(F.coalesce(F.col("socialEngagementType"), F.lit(""))))
        updated_df = updated_df.withColumn(
            "is_socially_engaged",
            F.when(
                normalized_social.isin("not socially engaged", STANDARD_UNKNOWN_VALUE, ""),
                F.lit(0),
            ).otherwise(F.lit(1)).cast("int"),
        )

    return updated_df


train_sessions_column_policy_df = add_sparse_indicator_flags(train_sessions_placeholder_clean_df).persist(StorageLevel.DISK_ONLY)
test_sessions_column_policy_df = add_sparse_indicator_flags(test_sessions_placeholder_clean_df).persist(StorageLevel.DISK_ONLY)

ADDED_POLICY_FLAG_COLUMNS = [
    flag_column for flag_column in list(SPARSE_FLAG_SPECS.values()) + ["is_socially_engaged"]
    if flag_column in train_sessions_column_policy_df.columns
]

print("Added policy flag columns:", ADDED_POLICY_FLAG_COLUMNS)

Added policy flag columns: ['has_traffic_campaign', 'has_traffic_ad_content', 'has_referral_path', 'has_geo_metro', 'has_custom_dimension', 'is_socially_engaged']


#### Buoc 4.5: Validation flag va tao danh sach feature/exclude

Sau khi tao flag, cap nhat policy cho cac flag moi va tao cac list dung cho cac buoc model sau nay.

In [18]:
ADDED_FLAG_TREATMENTS = {
    flag_column: {
        "role": "binary_flag",
        "decision": "keep_as_feature",
        "model_usage": "candidate_feature",
        "clean_action": "created_from_sparse_column_keep_0_1",
        "reason": "Flag duoc tao de thay the raw sparse/high-cardinality column.",
    }
    for flag_column in ADDED_POLICY_FLAG_COLUMNS
}

COLUMN_TREATMENT_WITH_FLAGS = {**COLUMN_TREATMENT_OVERRIDES, **ADDED_FLAG_TREATMENTS}


def build_column_treatment_plan_with_flags(df):
    rows = []
    dtype_by_column = dict(df.dtypes)

    for column_name in df.columns:
        treatment = {**DEFAULT_TREATMENT, **COLUMN_TREATMENT_WITH_FLAGS.get(column_name, {})}
        rows.append({
            "column": column_name,
            "dtype": dtype_by_column[column_name],
            **treatment,
        })

    return spark.createDataFrame(rows)


column_treatment_plan_with_flags_df = build_column_treatment_plan_with_flags(train_sessions_column_policy_df)

flag_summary_exprs = [F.sum(F.col(column)).alias(column) for column in ADDED_POLICY_FLAG_COLUMNS]
if flag_summary_exprs:
    print("Train flag positive counts:")
    train_sessions_column_policy_df.agg(*flag_summary_exprs).show(truncate=False)

    print("Test flag positive counts:")
    test_sessions_column_policy_df.agg(*flag_summary_exprs).show(truncate=False)

MODEL_EXCLUDE_COLUMNS = [
    row["column"]
    for row in column_treatment_plan_with_flags_df
    .where(
        F.col("decision").isin("drop_from_model", "target_leakage", "keep_key", "keep_for_eda", "keep_flag_only")
        | F.col("model_usage").isin(
            "exclude",
            "exclude_direct_feature",
            "exclude_raw",
            "exclude_raw_use_clean",
            "exclude_raw_use_family",
            "target_only",
            "use_flag_not_raw",
            "use_existing_has_traffic_keyword_not_raw",
            "use_existing_has_gclid_not_raw",
        )
    )
    .select("column")
    .collect()
]

REVIEW_BEFORE_MODEL_COLUMNS = [
    row["column"]
    for row in column_treatment_plan_with_flags_df
    .where(F.col("decision") == "review_before_model")
    .select("column")
    .collect()
]

MODEL_FEATURE_CANDIDATE_COLUMNS = [
    row["column"]
    for row in column_treatment_plan_with_flags_df
    .where(F.col("model_usage") == "candidate_feature")
    .select("column")
    .collect()
]

EDA_KEEP_COLUMNS = [
    row["column"]
    for row in column_treatment_plan_with_flags_df
    .where(F.col("decision").isin("keep_for_eda", "review_before_model"))
    .select("column")
    .collect()
]

print("MODEL_EXCLUDE_COLUMNS:", MODEL_EXCLUDE_COLUMNS)
print("REVIEW_BEFORE_MODEL_COLUMNS:", REVIEW_BEFORE_MODEL_COLUMNS)
print("MODEL_FEATURE_CANDIDATE_COLUMNS:", MODEL_FEATURE_CANDIDATE_COLUMNS)
print("EDA_KEEP_COLUMNS:", EDA_KEEP_COLUMNS)

print("Column policy with flags:")
column_treatment_plan_with_flags_df.orderBy("decision", "role", "column").show(250, truncate=False)

Train flag positive counts:
+--------------------+----------------------+-----------------+-------------+--------------------+-------------------+
|has_traffic_campaign|has_traffic_ad_content|has_referral_path|has_geo_metro|has_custom_dimension|is_socially_engaged|
+--------------------+----------------------+-----------------+-------------+--------------------+-------------------+
|103691              |64709                 |565830           |388153       |1373713             |0                  |
+--------------------+----------------------+-----------------+-------------+--------------------+-------------------+

Test flag positive counts:
+--------------------+----------------------+-----------------+-------------+--------------------+-------------------+
|has_traffic_campaign|has_traffic_ad_content|has_referral_path|has_geo_metro|has_custom_dimension|is_socially_engaged|
+--------------------+----------------------+-----------------+-------------+--------------------+-------------

### Buoc 5: Tao session purchase source signal

Muc tieu: chuan hoa raw revenue/transaction cua session hien tai thanh source signal. Source signal nay chi dung de tao label future 30 ngay o cuoi notebook, khong dung lam input feature.

Output cua buoc nay:
- `session_revenue_signal`
- `session_has_purchase_signal`
- `session_purchase_signal_source`

Luu y: day chua phai target cua bai toan. Target that su se la `future_30d_has_purchase` va `future_30d_revenue`, duoc tao sau khi clean xong.

In [19]:
REVENUE_MICRO_DIVISOR = 1_000_000.0
LABEL_WINDOW_DAYS = 30

SESSION_REVENUE_SOURCE_COLUMNS = [
    "transaction_revenue",
    "total_transaction_revenue",
    "totals_transaction_revenue",
    "totals_total_transaction_revenue",
]
SESSION_LABEL_SOURCE_COLUMNS = [
    "totals_transactions",
    "transaction_revenue",
    "total_transaction_revenue",
    "totals_transaction_revenue",
    "totals_total_transaction_revenue",
]
SESSION_SOURCE_SIGNAL_COLUMNS = [
    "session_revenue_signal",
    "session_has_purchase_signal",
    "session_purchase_signal_source",
]


def revenue_source_as_currency_expr(df, column_name):
    value = F.coalesce(F.col(column_name).cast("double"), F.lit(0.0))
    if column_name.startswith("totals_"):
        return value / F.lit(REVENUE_MICRO_DIVISOR)
    return value


def session_revenue_signal_expr(df):
    expressions = [
        revenue_source_as_currency_expr(df, column_name)
        for column_name in SESSION_REVENUE_SOURCE_COLUMNS
        if column_name in df.columns
    ]
    return F.greatest(*expressions) if expressions else F.lit(0.0)


def add_session_purchase_source_signal(df):
    revenue_signal = session_revenue_signal_expr(df)
    transaction_count = F.coalesce(F.col("totals_transactions").cast("long"), F.lit(0))
    return (
        df
        .withColumn("session_revenue_signal", F.when(revenue_signal < 0, F.lit(0.0)).otherwise(revenue_signal).cast("double"))
        .withColumn(
            "session_has_purchase_signal",
            F.when((F.col("session_revenue_signal") > 0) | (transaction_count > 0), F.lit(1)).otherwise(F.lit(0)).cast("int"),
        )
        .withColumn(
            "session_purchase_signal_source",
            F.when(F.col("session_revenue_signal") > 0, F.lit("revenue_positive"))
            .when(transaction_count > 0, F.lit("transaction_positive_no_revenue"))
            .otherwise(F.lit("no_purchase_signal")),
        )
    )


train_sessions_signal_clean_df = add_session_purchase_source_signal(train_sessions_column_policy_df).persist(StorageLevel.DISK_ONLY)
test_sessions_signal_clean_df = add_session_purchase_source_signal(test_sessions_column_policy_df).persist(StorageLevel.DISK_ONLY)

source_signal_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "totals_transactions",
    "transaction_revenue",
    "total_transaction_revenue",
    "session_revenue_signal",
    "session_has_purchase_signal",
    "session_purchase_signal_source",
]
train_sessions_signal_clean_df.select([column for column in source_signal_preview_columns if column in train_sessions_signal_clean_df.columns]).show(10, truncate=False)

+-------------------+----------+------------+-------------------+-------------------+-------------------------+----------------------+---------------------------+------------------------------+
|fullVisitorId      |visit_id  |session_date|totals_transactions|transaction_revenue|total_transaction_revenue|session_revenue_signal|session_has_purchase_signal|session_purchase_signal_source|
+-------------------+----------+------------+-------------------+-------------------+-------------------------+----------------------+---------------------------+------------------------------+
|0000040862739425590|1486836571|2017-02-11  |0                  |0.0                |0.0                      |0.0                   |0                          |no_purchase_signal            |
|0000174453501096099|1520448910|2018-03-07  |0                  |0.0                |0.0                      |0.0                   |0                          |no_purchase_signal            |
|000020731284570628 |150720470

In [20]:
def build_source_signal_summary(name, df):
    total_rows = df.count()
    row = df.agg(
        F.sum("session_has_purchase_signal").alias("purchase_signal_sessions"),
        F.sum(F.when(F.col("session_revenue_signal") > 0, 1).otherwise(0)).alias("revenue_positive_sessions"),
        F.sum(F.when(F.col("session_revenue_signal") < 0, 1).otherwise(0)).alias("negative_revenue_signal_sessions"),
        F.min("session_revenue_signal").alias("revenue_signal_min"),
        F.max("session_revenue_signal").alias("revenue_signal_max"),
    ).collect()[0]
    return {
        "dataset": name,
        "row_count": total_rows,
        "purchase_signal_sessions": int(row["purchase_signal_sessions"] or 0),
        "purchase_signal_rate_pct": round((row["purchase_signal_sessions"] or 0) / total_rows * 100, 4) if total_rows else 0.0,
        "revenue_positive_sessions": int(row["revenue_positive_sessions"] or 0),
        "negative_revenue_signal_sessions": int(row["negative_revenue_signal_sessions"] or 0),
        "revenue_signal_min": float(row["revenue_signal_min"] or 0.0),
        "revenue_signal_max": float(row["revenue_signal_max"] or 0.0),
    }


source_signal_summary_rows = [
    build_source_signal_summary("train_sessions", train_sessions_signal_clean_df),
    build_source_signal_summary("test_sessions", test_sessions_signal_clean_df),
]
source_signal_summary_df = spark.createDataFrame(source_signal_summary_rows)
source_signal_summary_df.show(truncate=False)

if any(row["negative_revenue_signal_sessions"] > 0 for row in source_signal_summary_rows):
    raise AssertionError(f"Source signal validation failed: {source_signal_summary_rows}")

SESSION_SOURCE_SIGNAL_POLICY_OVERRIDES = {
    column_name: {
        "role": "future_label_source_signal",
        "decision": "exclude_from_model",
        "model_usage": "exclude",
        "clean_action": "keep_for_future_30d_label_creation_only",
        "reason": "Tin hieu mua hang/doanh thu cua session hien tai, chi dung de tao future label va debug.",
    }
    for column_name in SESSION_LABEL_SOURCE_COLUMNS + SESSION_SOURCE_SIGNAL_COLUMNS
}
COLUMN_TREATMENT_WITH_SOURCE_SIGNAL = {**COLUMN_TREATMENT_WITH_FLAGS, **SESSION_SOURCE_SIGNAL_POLICY_OVERRIDES}
LABEL_SOURCE_AND_LEAKAGE_COLUMNS = sorted(set([column for column in SESSION_LABEL_SOURCE_COLUMNS + SESSION_SOURCE_SIGNAL_COLUMNS if column in train_sessions_signal_clean_df.columns]))
MODEL_EXCLUDE_COLUMNS = sorted(set(MODEL_EXCLUDE_COLUMNS + LABEL_SOURCE_AND_LEAKAGE_COLUMNS))
MODEL_FEATURE_CANDIDATE_COLUMNS = [column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column not in MODEL_EXCLUDE_COLUMNS]
print("LABEL_SOURCE_AND_LEAKAGE_COLUMNS:", LABEL_SOURCE_AND_LEAKAGE_COLUMNS)

+--------------+--------------------------------+------------------------+------------------------+-------------------------+------------------+------------------+---------+
|dataset       |negative_revenue_signal_sessions|purchase_signal_rate_pct|purchase_signal_sessions|revenue_positive_sessions|revenue_signal_max|revenue_signal_min|row_count|
+--------------+--------------------------------+------------------------+------------------------+-------------------------+------------------+------------------+---------+
|train_sessions|0                               |1.0874                  |18557                   |18512                    |47082.06          |0.0               |1706613  |
|test_sessions |0                               |1.5714                  |6303                    |4594                     |30174.94          |0.0               |401112   |
+--------------+--------------------------------+------------------------+------------------------+-------------------------+-----

### Buoc 6: Kiem tra quality cua source signal

Muc tieu: ghi ro cac truong hop source signal can review truoc khi tao future label.

In [21]:
SOURCE_SIGNAL_QUALITY_COLUMNS = [
    "source_signal_quality_status",
    "source_signal_quality_decision",
    "is_revenue_positive_signal",
    "is_transaction_only_signal",
]


def add_source_signal_quality_columns(df):
    has_revenue = F.col("session_revenue_signal") > 0
    has_transaction = F.coalesce(F.col("totals_transactions").cast("long"), F.lit(0)) > 0
    return (
        df
        .withColumn("is_revenue_positive_signal", F.when(has_revenue, 1).otherwise(0).cast("int"))
        .withColumn("is_transaction_only_signal", F.when((~has_revenue) & has_transaction, 1).otherwise(0).cast("int"))
        .withColumn(
            "source_signal_quality_status",
            F.when(has_revenue, F.lit("revenue_positive_signal"))
            .when((~has_revenue) & has_transaction, F.lit("transaction_only_signal_no_revenue"))
            .otherwise(F.lit("no_purchase_signal")),
        )
        .withColumn(
            "source_signal_quality_decision",
            F.when(has_revenue, F.lit("use_as_future_purchase_and_revenue_event"))
            .when((~has_revenue) & has_transaction, F.lit("use_as_future_purchase_event_revenue_zero"))
            .otherwise(F.lit("not_a_purchase_event")),
        )
    )


train_sessions_signal_quality_df = add_source_signal_quality_columns(train_sessions_signal_clean_df).persist(StorageLevel.DISK_ONLY)
test_sessions_signal_quality_df = add_source_signal_quality_columns(test_sessions_signal_clean_df).persist(StorageLevel.DISK_ONLY)

for dataset_name, dataset_df in [("train_sessions", train_sessions_signal_quality_df), ("test_sessions", test_sessions_signal_quality_df)]:
    print(dataset_name)
    dataset_df.groupBy("source_signal_quality_status", "source_signal_quality_decision").count().orderBy(F.desc("count")).show(20, truncate=False)

SOURCE_SIGNAL_QUALITY_POLICY_OVERRIDES = {
    column_name: {
        "role": "source_signal_quality_debug",
        "decision": "exclude_from_model",
        "model_usage": "exclude",
        "clean_action": "keep_for_future_label_debug_only",
        "reason": "Cot quality duoc sinh ra tu source signal, khong dung lam input feature.",
    }
    for column_name in SOURCE_SIGNAL_QUALITY_COLUMNS
}
COLUMN_TREATMENT_WITH_SOURCE_SIGNAL_QUALITY = {**COLUMN_TREATMENT_WITH_SOURCE_SIGNAL, **SOURCE_SIGNAL_QUALITY_POLICY_OVERRIDES}
MODEL_EXCLUDE_COLUMNS = sorted(set(MODEL_EXCLUDE_COLUMNS + SOURCE_SIGNAL_QUALITY_COLUMNS))
MODEL_FEATURE_CANDIDATE_COLUMNS = [column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column not in MODEL_EXCLUDE_COLUMNS]

train_sessions
+----------------------------------+-----------------------------------------+-------+
|source_signal_quality_status      |source_signal_quality_decision           |count  |
+----------------------------------+-----------------------------------------+-------+
|no_purchase_signal                |not_a_purchase_event                     |1688056|
|revenue_positive_signal           |use_as_future_purchase_and_revenue_event |18512  |
|transaction_only_signal_no_revenue|use_as_future_purchase_event_revenue_zero|45     |
+----------------------------------+-----------------------------------------+-------+

test_sessions
+----------------------------------+-----------------------------------------+------+
|source_signal_quality_status      |source_signal_quality_decision           |count |
+----------------------------------+-----------------------------------------+------+
|no_purchase_signal                |not_a_purchase_event                     |394809|
|revenue_positive

### Buoc 7: Chuan hoa categorical values

Muc tieu: tao ban categorical sach hon cho EDA/model, lowercase + trim va gom rare category thanh `other` dua tren train mapping.

In [22]:
CATEGORY_COLUMNS_TO_MODEL = [
    "traffic_source_clean",
    "traffic_medium_clean",
    "traffic_channel_type",
    "browser_family",
    "os_family",
    "geo_country",
    "device_category",
    "channelGrouping",
]
RARE_CATEGORY_MIN_COUNT = 100
RARE_CATEGORY_MIN_PCT = 0.01
CATEGORY_MODEL_SUFFIX = "_model"
RARE_CATEGORY_VALUE = "other"


def category_model_column_name(column_name):
    return f"{column_name}{CATEGORY_MODEL_SUFFIX}"


def clean_category_text_expr(column_name):
    normalized_value = F.lower(F.trim(F.coalesce(F.col(column_name).cast("string"), F.lit(STANDARD_UNKNOWN_VALUE))))
    return F.when(normalized_value.isin(CONFIRMED_PLACEHOLDER_VALUES) | (normalized_value == ""), F.lit(STANDARD_UNKNOWN_VALUE)).otherwise(normalized_value)


category_columns_to_model = existing_columns(train_sessions_signal_quality_df, CATEGORY_COLUMNS_TO_MODEL)
category_model_columns = [category_model_column_name(column) for column in category_columns_to_model]

In [23]:
def build_category_keep_mapping(train_df, columns):
    total_rows = train_df.count()
    mapping_rows = []
    distribution_dfs = []
    for column_name in columns:
        model_column = category_model_column_name(column_name)
        distribution_df = (
            train_df.select(clean_category_text_expr(column_name).alias("category_value"))
            .groupBy("category_value")
            .count()
            .withColumn("column", F.lit(column_name))
            .withColumn("model_column", F.lit(model_column))
            .withColumn("pct", F.round(F.col("count") / F.lit(total_rows) * 100, 6))
            .withColumn("category_decision", F.when((F.col("count") >= F.lit(RARE_CATEGORY_MIN_COUNT)) & (F.col("pct") >= F.lit(RARE_CATEGORY_MIN_PCT)), F.lit("keep")).otherwise(F.lit("map_to_other")))
        )
        distribution_dfs.append(distribution_df)
        keep_values = [row["category_value"] for row in distribution_df.where(F.col("category_decision") == "keep").select("category_value").collect()]
        mapping_rows.append({"column": column_name, "model_column": model_column, "keep_values": keep_values, "keep_value_count": len(keep_values)})
    mapping_distribution_df = distribution_dfs[0]
    for one_df in distribution_dfs[1:]:
        mapping_distribution_df = mapping_distribution_df.unionByName(one_df)
    return mapping_rows, mapping_distribution_df


category_keep_mapping_rows, category_train_distribution_df = build_category_keep_mapping(train_sessions_signal_quality_df, category_columns_to_model)
spark.createDataFrame(category_keep_mapping_rows).select("column", "model_column", "keep_value_count").orderBy("column").show(truncate=False)

+--------------------+--------------------------+----------------+
|column              |model_column              |keep_value_count|
+--------------------+--------------------------+----------------+
|browser_family      |browser_family_model      |6               |
|channelGrouping     |channelGrouping_model     |7               |
|device_category     |device_category_model     |3               |
|geo_country         |geo_country_model         |126             |
|os_family           |os_family_model           |6               |
|traffic_channel_type|traffic_channel_type_model|5               |
|traffic_medium_clean|traffic_medium_clean_model|6               |
|traffic_source_clean|traffic_source_clean_model|43              |
+--------------------+--------------------------+----------------+



In [24]:
def apply_category_model_mapping(df, mapping_rows):
    updated_df = df
    for row in mapping_rows:
        source_column = row["column"]
        model_column = row["model_column"]
        keep_values = row["keep_values"]
        cleaned_value = clean_category_text_expr(source_column)
        updated_df = updated_df.withColumn(model_column, F.when(cleaned_value.isin(keep_values), cleaned_value).otherwise(F.lit(RARE_CATEGORY_VALUE)))
    return updated_df


train_sessions_category_clean_df = apply_category_model_mapping(train_sessions_signal_quality_df, category_keep_mapping_rows).persist(StorageLevel.DISK_ONLY)
test_sessions_category_clean_df = apply_category_model_mapping(test_sessions_signal_quality_df, category_keep_mapping_rows).persist(StorageLevel.DISK_ONLY)

CATEGORY_MODEL_POLICY_OVERRIDES = {
    model_column: {"role": "model_ready_categorical", "decision": "keep_as_feature", "model_usage": "candidate_feature", "clean_action": "lower_trim_group_rare_to_other", "reason": "Cot categorical da duoc chuan hoa text va gom rare category bang train mapping."}
    for model_column in category_model_columns
}
COLUMN_TREATMENT_WITH_CATEGORY_MODEL = {**COLUMN_TREATMENT_WITH_SOURCE_SIGNAL_QUALITY, **CATEGORY_MODEL_POLICY_OVERRIDES}
MODEL_FEATURE_CANDIDATE_COLUMNS = sorted(set([column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column in train_sessions_category_clean_df.columns] + category_model_columns))
MODEL_FEATURE_CANDIDATE_COLUMNS = [column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column not in MODEL_EXCLUDE_COLUMNS]

### Buoc 8: Tao cot phu phuc vu EDA/model

Muc tieu: tao cac cot tien dung khong sinh tu target. Cac cot target future se duoc tao o buoc sau.

In [25]:
DERIVED_EDA_MODEL_COLUMNS = ["is_bounce", "has_pageviews", "session_month_key"]
MODEL_READY_DERIVED_FEATURE_COLUMNS = ["is_bounce", "has_pageviews"]


def add_derived_eda_model_columns(df):
    return (
        df
        .withColumn("is_bounce", F.when(F.coalesce(F.col("totals_bounces").cast("int"), F.lit(0)) == 1, 1).otherwise(0).cast("int"))
        .withColumn("has_pageviews", F.when(F.coalesce(F.col("totals_pageviews").cast("int"), F.lit(0)) > 0, 1).otherwise(0).cast("int"))
        .withColumn("session_month_key", F.date_format(F.col("session_date"), "yyyy-MM"))
    )


train_sessions_feature_ready_df = add_derived_eda_model_columns(train_sessions_category_clean_df).persist(StorageLevel.DISK_ONLY)
test_sessions_feature_ready_df = add_derived_eda_model_columns(test_sessions_category_clean_df).persist(StorageLevel.DISK_ONLY)

DERIVED_COLUMN_POLICY_OVERRIDES = {
    "is_bounce": {"role": "binary_behavior_feature", "decision": "keep_as_feature", "model_usage": "candidate_feature", "clean_action": "created_from_totals_bounces", "reason": "Flag bounce."},
    "has_pageviews": {"role": "binary_behavior_feature", "decision": "keep_as_feature", "model_usage": "candidate_feature", "clean_action": "created_from_totals_pageviews", "reason": "Flag co pageviews."},
    "session_month_key": {"role": "time_group_key", "decision": "keep_for_eda", "model_usage": "review_before_model", "clean_action": "use_for_groupby_and_time_series_eda", "reason": "Key thang cho EDA."},
}
COLUMN_TREATMENT_WITH_DERIVED_COLUMNS = {**COLUMN_TREATMENT_WITH_CATEGORY_MODEL, **DERIVED_COLUMN_POLICY_OVERRIDES}
MODEL_FEATURE_CANDIDATE_COLUMNS = sorted(set([column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column in train_sessions_feature_ready_df.columns] + MODEL_READY_DERIVED_FEATURE_COLUMNS))
MODEL_FEATURE_CANDIDATE_COLUMNS = [column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column not in MODEL_EXCLUDE_COLUMNS]

### Buoc 9: Tao future 30-day label

Muc tieu: sau khi du lieu da clean day du, tao label cho bai toan: du doan khach hang co mua hang trong 30 ngay tiep theo ke tu session hien tai hay khong, va neu co thi doanh thu la bao nhieu.

De tranh JVM bi qua tai:
- Khong self-join anchor sessions voi purchase events.
- Khong persist dataframe wide sau khi join label.
- Ghi label base cuc hep ra Parquet tam, doc lai de cat lineage, roi moi tinh window `rangeBetween` theo tung `fullVisitorId`.


In [ ]:
FUTURE_LABEL_COLUMNS = [
    "future_30d_has_purchase",
    "future_30d_revenue",
    "future_30d_purchase_count",
    "future_30d_first_purchase_timestamp",
    "days_to_first_purchase",
    "has_full_30d_label_window",
]

FUTURE_LABEL_WORK_DIR = PARQUET_DIR / "_tmp_future_30d_label_work"
FUTURE_LABEL_REPARTITION_COUNT = 8
FUTURE_LABEL_MAX_RECORDS_PER_FILE = 500_000


def prepare_future_label_base(df):
    return (
        df
        .select(
            "fullVisitorId",
            "visit_id",
            F.col("visit_start_time").cast("long").alias("visit_start_time_long"),
            "visit_start_timestamp",
            "session_date",
            F.coalesce(F.col("session_has_purchase_signal").cast("int"), F.lit(0)).alias("session_has_purchase_signal"),
            F.coalesce(F.col("session_revenue_signal").cast("double"), F.lit(0.0)).alias("session_revenue_signal"),
        )
        .where(F.col("fullVisitorId").isNotNull() & F.col("visit_id").isNotNull() & F.col("visit_start_time_long").isNotNull())
    )


def build_future_30d_label_frame_from_base(label_base_df):
    """Tinh future label tren bang hep da materialize de tranh lineage dai va join rong."""
    max_session_date = label_base_df.agg(F.max("session_date").alias("max_session_date")).collect()[0]["max_session_date"]
    window_seconds = LABEL_WINDOW_DAYS * 24 * 60 * 60
    future_window = (
        Window
        .partitionBy("fullVisitorId")
        .orderBy(F.col("visit_start_time_long"))
        .rangeBetween(1, window_seconds)
    )

    purchase_count_expr = F.when(F.col("session_has_purchase_signal") == 1, F.lit(1)).otherwise(F.lit(0))
    purchase_revenue_expr = F.when(F.col("session_has_purchase_signal") == 1, F.col("session_revenue_signal")).otherwise(F.lit(0.0))
    purchase_timestamp_expr = F.when(F.col("session_has_purchase_signal") == 1, F.col("visit_start_timestamp"))

    return (
        label_base_df
        .withColumn("future_30d_purchase_count_raw", F.coalesce(F.sum(purchase_count_expr).over(future_window), F.lit(0)).cast("int"))
        .withColumn("future_30d_revenue_raw", F.coalesce(F.sum(purchase_revenue_expr).over(future_window), F.lit(0.0)).cast("double"))
        .withColumn("future_30d_first_purchase_timestamp_raw", F.min(purchase_timestamp_expr).over(future_window))
        .withColumn(
            "has_full_30d_label_window",
            F.when(F.col("session_date") <= F.date_sub(F.lit(max_session_date), LABEL_WINDOW_DAYS), 1)
            .otherwise(0)
            .cast("int"),
        )
        .withColumn(
            "future_30d_has_purchase",
            F.when(F.col("has_full_30d_label_window") == 0, F.lit(None).cast("int"))
            .when(F.col("future_30d_purchase_count_raw") > 0, F.lit(1))
            .otherwise(F.lit(0))
            .cast("int"),
        )
        .withColumn(
            "future_30d_revenue",
            F.when(F.col("has_full_30d_label_window") == 0, F.lit(None).cast("double"))
            .otherwise(F.col("future_30d_revenue_raw"))
            .cast("double"),
        )
        .withColumn(
            "future_30d_purchase_count",
            F.when(F.col("has_full_30d_label_window") == 0, F.lit(None).cast("int"))
            .otherwise(F.col("future_30d_purchase_count_raw"))
            .cast("int"),
        )
        .withColumn(
            "future_30d_first_purchase_timestamp",
            F.when(F.col("has_full_30d_label_window") == 0, F.lit(None).cast("timestamp"))
            .otherwise(F.col("future_30d_first_purchase_timestamp_raw")),
        )
        .withColumn(
            "days_to_first_purchase",
            F.when(F.col("future_30d_first_purchase_timestamp").isNull(), F.lit(None).cast("int"))
            .otherwise(F.datediff(F.to_date("future_30d_first_purchase_timestamp"), F.col("session_date")))
            .cast("int"),
        )
        .select("fullVisitorId", "visit_id", *FUTURE_LABEL_COLUMNS)
    )


def materialize_future_30d_labels(df, dataset_name):
    base_path = FUTURE_LABEL_WORK_DIR / dataset_name / "label_base"
    label_path = FUTURE_LABEL_WORK_DIR / dataset_name / "future_30d_labels"

    print(f"Writing narrow label base: {base_path}")
    (
        prepare_future_label_base(df)
        .repartition(FUTURE_LABEL_REPARTITION_COUNT, "fullVisitorId")
        .write
        .mode("overwrite")
        .option("maxRecordsPerFile", FUTURE_LABEL_MAX_RECORDS_PER_FILE)
        .parquet(str(base_path))
    )

    label_base_df = spark.read.parquet(str(base_path)).repartition(FUTURE_LABEL_REPARTITION_COUNT, "fullVisitorId")
    print(f"Writing materialized future labels: {label_path}")
    (
        build_future_30d_label_frame_from_base(label_base_df)
        .write
        .mode("overwrite")
        .option("maxRecordsPerFile", FUTURE_LABEL_MAX_RECORDS_PER_FILE)
        .parquet(str(label_path))
    )

    return spark.read.parquet(str(label_path))


train_future_30d_label_df = materialize_future_30d_labels(train_sessions_feature_ready_df, "train_sessions")
test_future_30d_label_df = materialize_future_30d_labels(test_sessions_feature_ready_df, "test_sessions")

# Lazy join only for downstream writing. Do not persist or show this wide dataframe in Step 9.
train_sessions_labeled_30d_df = train_sessions_feature_ready_df.join(train_future_30d_label_df, on=["fullVisitorId", "visit_id"], how="left")
test_sessions_labeled_30d_df = test_sessions_feature_ready_df.join(test_future_30d_label_df, on=["fullVisitorId", "visit_id"], how="left")

train_future_30d_label_df.show(20, truncate=False)


In [ ]:
def build_future_label_summary(name, df):
    total_rows = df.count()
    row = df.agg(
        F.sum("has_full_30d_label_window").alias("sessions_with_full_window"),
        F.sum(F.when(F.col("has_full_30d_label_window") == 0, 1).otherwise(0)).alias("sessions_without_full_window"),
        F.sum(F.when(F.col("future_30d_has_purchase") == 1, 1).otherwise(0)).alias("future_purchase_sessions"),
        F.sum(F.when(F.col("future_30d_revenue") > 0, 1).otherwise(0)).alias("future_revenue_positive_sessions"),
        F.min("future_30d_revenue").alias("future_revenue_min"),
        F.max("future_30d_revenue").alias("future_revenue_max"),
        F.avg("future_30d_revenue").alias("future_revenue_avg"),
    ).collect()[0]
    with_window = row["sessions_with_full_window"] or 0
    future_purchase = row["future_purchase_sessions"] or 0
    return {
        "dataset": name,
        "row_count": total_rows,
        "label_window_days": LABEL_WINDOW_DAYS,
        "sessions_with_full_window": int(with_window),
        "sessions_without_full_window": int(row["sessions_without_full_window"] or 0),
        "future_purchase_sessions": int(future_purchase),
        "future_purchase_rate_on_full_window_pct": round(future_purchase / with_window * 100, 4) if with_window else 0.0,
        "future_revenue_positive_sessions": int(row["future_revenue_positive_sessions"] or 0),
        "future_revenue_min": float(row["future_revenue_min"] or 0.0),
        "future_revenue_max": float(row["future_revenue_max"] or 0.0),
        "future_revenue_avg": float(row["future_revenue_avg"] or 0.0),
    }


future_label_summary_rows = [
    build_future_label_summary("train_sessions", train_future_30d_label_df),
    build_future_label_summary("test_sessions", test_future_30d_label_df),
]
future_label_summary_df = spark.createDataFrame(future_label_summary_rows)
future_label_summary_df.show(truncate=False)

FUTURE_LABEL_POLICY_OVERRIDES = {
    "future_30d_has_purchase": {"role": "future_30d_target_label", "decision": "target", "model_usage": "target_only", "clean_action": "use_as_classification_target", "reason": "Label mua hang trong 30 ngay tiep theo."},
    "future_30d_revenue": {"role": "future_30d_target_revenue", "decision": "target", "model_usage": "target_only", "clean_action": "use_as_regression_target", "reason": "Tong revenue trong 30 ngay tiep theo."},
    "future_30d_purchase_count": {"role": "future_30d_target_debug", "decision": "target_debug", "model_usage": "exclude", "clean_action": "keep_for_label_debug", "reason": "So purchase event trong window."},
    "future_30d_first_purchase_timestamp": {"role": "future_30d_target_debug", "decision": "target_debug", "model_usage": "exclude", "clean_action": "keep_for_label_debug", "reason": "Thoi diem purchase dau tien trong window."},
    "days_to_first_purchase": {"role": "future_30d_target_debug", "decision": "target_debug", "model_usage": "exclude", "clean_action": "keep_for_label_debug", "reason": "Khoang cach den purchase dau tien."},
    "has_full_30d_label_window": {"role": "label_window_flag", "decision": "training_filter", "model_usage": "exclude", "clean_action": "use_to_filter_supervised_training", "reason": "Flag session co du future 30 ngay."},
}
COLUMN_TREATMENT_WITH_FUTURE_LABEL = {**COLUMN_TREATMENT_WITH_DERIVED_COLUMNS, **FUTURE_LABEL_POLICY_OVERRIDES}
MODEL_TARGET_COLUMNS = ["future_30d_has_purchase", "future_30d_revenue"]
FUTURE_LABEL_AND_DEBUG_COLUMNS = FUTURE_LABEL_COLUMNS
MODEL_EXCLUDE_COLUMNS = sorted(set(MODEL_EXCLUDE_COLUMNS + FUTURE_LABEL_AND_DEBUG_COLUMNS))
MODEL_FEATURE_CANDIDATE_COLUMNS = [column for column in MODEL_FEATURE_CANDIDATE_COLUMNS if column not in MODEL_EXCLUDE_COLUMNS]


### Buoc 10: Ghi clean 30d label Parquet theo column chunks

Muc tieu: ghi output clean da co label future 30 ngay ra Parquet rieng, theo column chunks de tranh JVM qua tai.

Output:
- `data_pyspark_parquet/train_sessions_clean_30d_label`
- `data_pyspark_parquet/test_sessions_clean_30d_label`

In [28]:
import shutil
import json
from datetime import datetime, timezone

TRAIN_CLEAN_30D_LABEL_PARQUET_PATH = PARQUET_DIR / "train_sessions_clean_30d_label"
TEST_CLEAN_30D_LABEL_PARQUET_PATH = PARQUET_DIR / "test_sessions_clean_30d_label"
TRAIN_CLEAN_PARQUET_PATH = TRAIN_CLEAN_30D_LABEL_PARQUET_PATH
TEST_CLEAN_PARQUET_PATH = TEST_CLEAN_30D_LABEL_PARQUET_PATH

CLEAN_WRITE_MODE = "overwrite"
CLEAN_REPARTITION_COUNT = None
CLEAN_MAX_RECORDS_PER_FILE = 500_000
CLEAN_PARTITION_COLUMNS = ["session_year", "session_month"]
CLEAN_JOIN_KEY_COLUMNS = ["fullVisitorId", "visit_id"]
CLEAN_COLUMN_CHUNK_SIZE = 18


def unique_preserve_order(values):
    seen = set()
    result = []
    for value in values:
        if value not in seen:
            seen.add(value)
            result.append(value)
    return result


train_clean_columns = unique_preserve_order(train_sessions_feature_ready_df.columns + FUTURE_LABEL_COLUMNS)
test_clean_columns = unique_preserve_order(test_sessions_feature_ready_df.columns + FUTURE_LABEL_COLUMNS)
missing_in_test = sorted(set(train_clean_columns) - set(test_clean_columns))
missing_in_train = sorted(set(test_clean_columns) - set(train_clean_columns))
if missing_in_test or missing_in_train:
    raise ValueError(f"Train/test clean schema khong khop: missing_in_test={missing_in_test}, missing_in_train={missing_in_train}")

CLEAN_OUTPUT_COLUMNS = train_clean_columns
train_sessions_clean_df = train_sessions_labeled_30d_df.select(CLEAN_OUTPUT_COLUMNS)
test_sessions_clean_df = test_sessions_labeled_30d_df.select(CLEAN_OUTPUT_COLUMNS)


def build_clean_column_chunks(columns, chunk_size=CLEAN_COLUMN_CHUNK_SIZE):
    required_columns = unique_preserve_order(CLEAN_JOIN_KEY_COLUMNS + CLEAN_PARTITION_COLUMNS)
    chunk_source_columns = [column for column in columns if column not in required_columns]
    chunks = []
    for start_index in range(0, len(chunk_source_columns), chunk_size):
        chunk_index = len(chunks) + 1
        data_columns = chunk_source_columns[start_index:start_index + chunk_size]
        chunk_columns = unique_preserve_order(CLEAN_JOIN_KEY_COLUMNS + data_columns + CLEAN_PARTITION_COLUMNS)
        chunks.append({"chunk_id": f"chunk_{chunk_index:03d}", "data_columns": data_columns, "columns": chunk_columns, "column_count": len(chunk_columns), "data_column_count": len(data_columns)})
    return chunks


CLEAN_COLUMN_CHUNKS = build_clean_column_chunks(CLEAN_OUTPUT_COLUMNS)


def reset_clean_output_container(output_path):
    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)


def build_chunk_dataframe(feature_df, label_df, chunk_columns):
    requested_label_columns = [column for column in chunk_columns if column in FUTURE_LABEL_COLUMNS]
    if requested_label_columns:
        feature_columns = [column for column in chunk_columns if column not in requested_label_columns]
        left_columns = unique_preserve_order(CLEAN_JOIN_KEY_COLUMNS + feature_columns)
        right_columns = unique_preserve_order(CLEAN_JOIN_KEY_COLUMNS + requested_label_columns)
        return (
            feature_df
            .select(left_columns)
            .join(label_df.select(right_columns), on=CLEAN_JOIN_KEY_COLUMNS, how="left")
            .select(chunk_columns)
        )
    return feature_df.select(chunk_columns)


def write_clean_column_chunks(feature_df, label_df, output_path, chunks):
    reset_clean_output_container(output_path)
    for chunk in chunks:
        chunk_path = output_path / chunk["chunk_id"]
        chunk_df = build_chunk_dataframe(feature_df, label_df, chunk["columns"])
        if CLEAN_REPARTITION_COUNT is not None:
            chunk_df = chunk_df.repartition(CLEAN_REPARTITION_COUNT, *CLEAN_PARTITION_COLUMNS)
        print(f"Writing {chunk_path} with {len(chunk['columns'])} columns")
        chunk_df.write.mode(CLEAN_WRITE_MODE).option("maxRecordsPerFile", CLEAN_MAX_RECORDS_PER_FILE).partitionBy(*CLEAN_PARTITION_COLUMNS).parquet(str(chunk_path))


write_clean_column_chunks(train_sessions_feature_ready_df, train_future_30d_label_df, TRAIN_CLEAN_PARQUET_PATH, CLEAN_COLUMN_CHUNKS)
write_clean_column_chunks(test_sessions_feature_ready_df, test_future_30d_label_df, TEST_CLEAN_PARQUET_PATH, CLEAN_COLUMN_CHUNKS)


Writing g:\ds\data_pyspark_parquet\train_sessions_clean_30d_label\chunk_005 with 19 columns
Writing g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label\chunk_001 with 22 columns
Writing g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label\chunk_002 with 22 columns
Writing g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label\chunk_003 with 22 columns
Writing g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label\chunk_004 with 22 columns
Writing g:\ds\data_pyspark_parquet\test_sessions_clean_30d_label\chunk_005 with 19 columns


In [29]:
CLEAN_COLUMN_TREATMENT_PLAN_PATH = PARQUET_DIR / "clean_30d_label_column_treatment_plan"
CLEAN_CATEGORY_TRAIN_DISTRIBUTION_PATH = PARQUET_DIR / "clean_30d_label_category_train_distribution"
CLEAN_SESSIONS_MANIFEST_PATH = PARQUET_DIR / "clean_30d_label_sessions_manifest.json"


def build_column_treatment_plan_with_future_label(df):
    rows = []
    dtype_by_column = dict(df.dtypes)
    for column_name in df.columns:
        treatment = {**DEFAULT_TREATMENT, **COLUMN_TREATMENT_WITH_FUTURE_LABEL.get(column_name, {})}
        rows.append({"column": column_name, "dtype": dtype_by_column[column_name], **treatment})
    return spark.createDataFrame(rows)


column_treatment_plan_with_future_label_df = build_column_treatment_plan_with_future_label(train_sessions_clean_df)
column_treatment_plan_with_future_label_df.write.mode("overwrite").parquet(str(CLEAN_COLUMN_TREATMENT_PLAN_PATH))
category_train_distribution_df.write.mode("overwrite").parquet(str(CLEAN_CATEGORY_TRAIN_DISTRIBUTION_PATH))

clean_chunk_manifest_rows = [{**chunk, "train_chunk_path": str(TRAIN_CLEAN_PARQUET_PATH / chunk["chunk_id"]), "test_chunk_path": str(TEST_CLEAN_PARQUET_PATH / chunk["chunk_id"])} for chunk in CLEAN_COLUMN_CHUNKS]
clean_sessions_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "storage_layout": "column_chunks",
    "label_type": "future_window",
    "label_window_days": LABEL_WINDOW_DAYS,
    "train_clean_path": str(TRAIN_CLEAN_PARQUET_PATH),
    "test_clean_path": str(TEST_CLEAN_PARQUET_PATH),
    "column_treatment_plan_path": str(CLEAN_COLUMN_TREATMENT_PLAN_PATH),
    "category_train_distribution_path": str(CLEAN_CATEGORY_TRAIN_DISTRIBUTION_PATH),
    "partition_columns": CLEAN_PARTITION_COLUMNS,
    "join_key_columns": CLEAN_JOIN_KEY_COLUMNS,
    "column_chunks": clean_chunk_manifest_rows,
    "clean_output_column_count": len(CLEAN_OUTPUT_COLUMNS),
    "model_target_columns": MODEL_TARGET_COLUMNS,
    "model_feature_candidate_columns": MODEL_FEATURE_CANDIDATE_COLUMNS,
    "model_exclude_columns": MODEL_EXCLUDE_COLUMNS,
    "label_source_and_leakage_columns": LABEL_SOURCE_AND_LEAKAGE_COLUMNS,
    "future_label_and_debug_columns": FUTURE_LABEL_AND_DEBUG_COLUMNS,
    "category_model_columns": category_model_columns,
    "derived_eda_model_columns": DERIVED_EDA_MODEL_COLUMNS,
}
with open(CLEAN_SESSIONS_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(clean_sessions_manifest, file, ensure_ascii=False, indent=2, default=str)
print("Wrote clean 30d label manifest to:", CLEAN_SESSIONS_MANIFEST_PATH)

Wrote clean 30d label manifest to: g:\ds\data_pyspark_parquet\clean_30d_label_sessions_manifest.json


### Buoc 11: Validation sau cleaning va future 30d label

Muc tieu: kiem tra output clean da co label future 30 ngay, doc toi thieu cac cot can validation tu column chunks.

In [30]:
def find_chunks_for_requested_columns(requested_columns, chunks=CLEAN_COLUMN_CHUNKS):
    requested_set = set(requested_columns)
    key_partition_set = set(CLEAN_JOIN_KEY_COLUMNS + CLEAN_PARTITION_COLUMNS)
    non_key_requested = requested_set - key_partition_set
    selected_chunks = [chunk for chunk in chunks if non_key_requested.intersection(set(chunk["columns"]))]
    if not selected_chunks and chunks:
        selected_chunks = [chunks[0]]
    return selected_chunks


def read_clean_selected_columns(base_path, requested_columns, chunks=CLEAN_COLUMN_CHUNKS):
    requested_columns = unique_preserve_order(requested_columns)
    selected_chunks = find_chunks_for_requested_columns(requested_columns, chunks)
    result_df = None
    selected_so_far = set()
    for chunk in selected_chunks:
        chunk_path = base_path / chunk["chunk_id"]
        chunk_available_columns = set(chunk["columns"])
        requested_from_chunk = [column for column in requested_columns if column in chunk_available_columns and column not in selected_so_far]
        read_columns = unique_preserve_order(CLEAN_JOIN_KEY_COLUMNS + requested_from_chunk)
        one_chunk_df = spark.read.parquet(str(chunk_path)).select(read_columns)
        if result_df is None:
            result_df = one_chunk_df
        else:
            result_df = result_df.join(one_chunk_df, on=CLEAN_JOIN_KEY_COLUMNS, how="left")
        selected_so_far.update(requested_from_chunk)
    final_columns = [column for column in requested_columns if column in result_df.columns]
    return result_df.select(final_columns)


VALIDATION_IMPORTANT_COLUMNS = unique_preserve_order(
    CLEAN_JOIN_KEY_COLUMNS + CLEAN_PARTITION_COLUMNS + [
        "session_date",
        "future_30d_has_purchase",
        "future_30d_revenue",
        "future_30d_purchase_count",
        "has_full_30d_label_window",
        "session_month_key",
    ] + [column for column in category_model_columns if column in CLEAN_OUTPUT_COLUMNS]
)
train_clean_validation_df = read_clean_selected_columns(TRAIN_CLEAN_PARQUET_PATH, VALIDATION_IMPORTANT_COLUMNS).persist(StorageLevel.DISK_ONLY)
test_clean_validation_df = read_clean_selected_columns(TEST_CLEAN_PARQUET_PATH, VALIDATION_IMPORTANT_COLUMNS).persist(StorageLevel.DISK_ONLY)

In [31]:
def build_clean_duplicate_validation(name, original_df, clean_df):
    before_rows = original_df.count()
    after_rows = clean_df.count()
    key_counts_df = clean_df.groupBy(*CLEAN_JOIN_KEY_COLUMNS).count().persist(StorageLevel.DISK_ONLY)
    duplicate_extra_rows_after_clean = key_counts_df.where(F.col("count") > 1).agg(F.coalesce(F.sum(F.col("count") - F.lit(1)), F.lit(0)).cast("long").alias("extra_rows")).collect()[0]["extra_rows"]
    duplicate_session_after_clean = key_counts_df.where(F.col("count") > 1).count()
    key_counts_df.unpersist()
    return {"dataset": name, "before_rows": before_rows, "after_rows": after_rows, "removed_duplicate_rows": before_rows - after_rows, "duplicate_session_after_clean": duplicate_session_after_clean, "duplicate_extra_rows_after_clean": duplicate_extra_rows_after_clean}


clean_duplicate_validation_rows = [
    build_clean_duplicate_validation("train_sessions", train_sessions_df, train_clean_validation_df),
    build_clean_duplicate_validation("test_sessions", test_sessions_df, test_clean_validation_df),
]
spark.createDataFrame(clean_duplicate_validation_rows).show(truncate=False)
if any(row["duplicate_extra_rows_after_clean"] != 0 for row in clean_duplicate_validation_rows):
    raise AssertionError(f"Clean duplicate validation failed: {clean_duplicate_validation_rows}")

+----------+-----------+--------------+--------------------------------+-----------------------------+----------------------+
|after_rows|before_rows|dataset       |duplicate_extra_rows_after_clean|duplicate_session_after_clean|removed_duplicate_rows|
+----------+-----------+--------------+--------------------------------+-----------------------------+----------------------+
|1706613   |1708337    |train_sessions|0                               |0                            |1724                  |
|401112    |401589     |test_sessions |0                               |0                            |477                   |
+----------+-----------+--------------+--------------------------------+-----------------------------+----------------------+



In [32]:
VALIDATION_PLACEHOLDER_COLUMNS = [column for column in category_model_columns + ["session_month_key"] if column in VALIDATION_IMPORTANT_COLUMNS]


def build_clean_placeholder_validation(name, df, columns):
    total_rows = df.count()
    rows = []
    raw_placeholder_values = [value for value in NON_STANDARD_PLACEHOLDER_VALUES if value != ""] + [""]
    for column_name in columns:
        normalized_value = F.lower(F.trim(F.coalesce(F.col(column_name).cast("string"), F.lit(""))))
        stats = df.agg(F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias("null_count"), F.sum(F.when(normalized_value.isin(raw_placeholder_values), 1).otherwise(0)).alias("placeholder_count")).collect()[0]
        null_count = int(stats["null_count"] or 0)
        placeholder_count = int(stats["placeholder_count"] or 0)
        rows.append({"dataset": name, "column": column_name, "row_count": total_rows, "null_count": null_count, "placeholder_count": placeholder_count})
    return rows


clean_placeholder_validation_rows = build_clean_placeholder_validation("train_sessions", train_clean_validation_df, VALIDATION_PLACEHOLDER_COLUMNS) + build_clean_placeholder_validation("test_sessions", test_clean_validation_df, VALIDATION_PLACEHOLDER_COLUMNS)
spark.createDataFrame(clean_placeholder_validation_rows).show(100, truncate=False)
if any(row["null_count"] > 0 or row["placeholder_count"] > 0 for row in clean_placeholder_validation_rows):
    raise AssertionError(f"Clean placeholder validation failed: {clean_placeholder_validation_rows}")

+--------------------------+--------------+----------+-----------------+---------+
|column                    |dataset       |null_count|placeholder_count|row_count|
+--------------------------+--------------+----------+-----------------+---------+
|traffic_source_clean_model|train_sessions|0         |0                |1706613  |
|traffic_medium_clean_model|train_sessions|0         |0                |1706613  |
|traffic_channel_type_model|train_sessions|0         |0                |1706613  |
|browser_family_model      |train_sessions|0         |0                |1706613  |
|os_family_model           |train_sessions|0         |0                |1706613  |
|geo_country_model         |train_sessions|0         |0                |1706613  |
|device_category_model     |train_sessions|0         |0                |1706613  |
|channelGrouping_model     |train_sessions|0         |0                |1706613  |
|session_month_key         |train_sessions|0         |0                |1706613  |
|tra

In [33]:
def build_future_label_validation(name, df):
    total_rows = df.count()
    checks = {
        "invalid_future_30d_has_purchase": (~F.col("future_30d_has_purchase").isin(0, 1)) & F.col("future_30d_has_purchase").isNotNull(),
        "label_null_with_full_window": (F.col("has_full_30d_label_window") == 1) & F.col("future_30d_has_purchase").isNull(),
        "label_not_null_without_full_window": (F.col("has_full_30d_label_window") == 0) & F.col("future_30d_has_purchase").isNotNull(),
        "negative_future_30d_revenue": F.col("future_30d_revenue") < 0,
        "revenue_positive_but_label_not_1": (F.col("future_30d_revenue") > 0) & (F.col("future_30d_has_purchase") != 1),
    }
    rows = []
    for check_name, condition in checks.items():
        affected_rows = df.where(condition).count()
        rows.append({"dataset": name, "check_name": check_name, "affected_rows": affected_rows, "affected_pct": round(affected_rows / total_rows * 100, 4) if total_rows else 0.0, "status": "pass" if affected_rows == 0 else "fail"})
    return rows


future_label_validation_rows = build_future_label_validation("train_sessions", train_clean_validation_df) + build_future_label_validation("test_sessions", test_clean_validation_df)
spark.createDataFrame(future_label_validation_rows).show(truncate=False)
if any(row["status"] == "fail" for row in future_label_validation_rows):
    raise AssertionError(f"Future label validation failed: {future_label_validation_rows}")

for dataset_name, dataset_df in [("train_sessions", train_clean_validation_df), ("test_sessions", test_clean_validation_df)]:
    print(dataset_name)
    dataset_df.groupBy("has_full_30d_label_window", "future_30d_has_purchase").count().orderBy("has_full_30d_label_window", "future_30d_has_purchase").show(truncate=False)

+------------+-------------+----------------------------------+--------------+------+
|affected_pct|affected_rows|check_name                        |dataset       |status|
+------------+-------------+----------------------------------+--------------+------+
|0.0         |0            |invalid_future_30d_has_purchase   |train_sessions|pass  |
|0.0         |0            |label_null_with_full_window       |train_sessions|pass  |
|0.0         |0            |label_not_null_without_full_window|train_sessions|pass  |
|0.0         |0            |negative_future_30d_revenue       |train_sessions|pass  |
|0.0         |0            |revenue_positive_but_label_not_1  |train_sessions|pass  |
|0.0         |0            |invalid_future_30d_has_purchase   |test_sessions |pass  |
|0.0         |0            |label_null_with_full_window       |test_sessions |pass  |
|0.0         |0            |label_not_null_without_full_window|test_sessions |pass  |
|0.0         |0            |negative_future_30d_revenu

In [34]:
original_data_integrity_rows = [
    {"dataset": "train_sessions_original", "path": str(TRAIN_PARQUET_PATH), "path_exists": TRAIN_PARQUET_PATH.exists(), "success_marker_exists": (TRAIN_PARQUET_PATH / "_SUCCESS").exists(), "row_count": train_sessions_df.count(), "column_count": len(train_sessions_df.columns)},
    {"dataset": "test_sessions_original", "path": str(TEST_PARQUET_PATH), "path_exists": TEST_PARQUET_PATH.exists(), "success_marker_exists": (TEST_PARQUET_PATH / "_SUCCESS").exists(), "row_count": test_sessions_df.count(), "column_count": len(test_sessions_df.columns)},
]
spark.createDataFrame(original_data_integrity_rows).show(truncate=False)
for row in original_data_integrity_rows:
    if not row["path_exists"] or not row["success_marker_exists"]:
        raise AssertionError(f"Du lieu goc khong con nguyen ven: {row}")

CLEAN_VALIDATION_REPORT_PATH = PARQUET_DIR / "clean_30d_label_validation_report"
validation_report_rows = []
for row in clean_duplicate_validation_rows:
    validation_report_rows.append({"dataset": row["dataset"], "check_name": "duplicate_session_after_clean", "metric_name": "duplicate_extra_rows_after_clean", "metric_value": float(row["duplicate_extra_rows_after_clean"]), "status": "pass" if row["duplicate_extra_rows_after_clean"] == 0 else "fail", "detail": json.dumps(row, ensure_ascii=False, default=str)})
for row in clean_placeholder_validation_rows:
    affected = row["null_count"] + row["placeholder_count"]
    validation_report_rows.append({"dataset": row["dataset"], "check_name": f"placeholder_{row['column']}", "metric_name": "null_plus_placeholder_count", "metric_value": float(affected), "status": "pass" if affected == 0 else "fail", "detail": json.dumps(row, ensure_ascii=False, default=str)})
for row in future_label_validation_rows:
    validation_report_rows.append({"dataset": row["dataset"], "check_name": row["check_name"], "metric_name": "affected_rows", "metric_value": float(row["affected_rows"]), "status": row["status"], "detail": json.dumps(row, ensure_ascii=False, default=str)})
spark.createDataFrame(validation_report_rows).write.mode("overwrite").parquet(str(CLEAN_VALIDATION_REPORT_PATH))
print("Wrote clean 30d label validation report to:", CLEAN_VALIDATION_REPORT_PATH)

+------------+-----------------------+-----------------------------------------+-----------+---------+---------------------+
|column_count|dataset                |path                                     |path_exists|row_count|success_marker_exists|
+------------+-----------------------+-----------------------------------------+-----------+---------+---------------------+
|61          |train_sessions_original|g:\ds\data_pyspark_parquet\train_sessions|true       |1708337  |true                 |
|61          |test_sessions_original |g:\ds\data_pyspark_parquet\test_sessions |true       |401589   |true                 |
+------------+-----------------------+-----------------------------------------+-----------+---------+---------------------+

Wrote clean 30d label validation report to: g:\ds\data_pyspark_parquet\clean_30d_label_validation_report


# Tong ket

Notebook nay da hoan thanh pipeline clean session-level data va tao label future 30 ngay.

Output clean co label:
- `data_pyspark_parquet/train_sessions_clean_30d_label`
- `data_pyspark_parquet/test_sessions_clean_30d_label`

Artifact ho tro:
- `data_pyspark_parquet/clean_30d_label_sessions_manifest.json`
- `data_pyspark_parquet/clean_30d_label_column_treatment_plan`
- `data_pyspark_parquet/clean_30d_label_category_train_distribution`
- `data_pyspark_parquet/clean_30d_label_validation_report`

Ket qua mong doi sau khi chay het notebook:
- Duplicate session key sau clean = 0.
- `future_30d_has_purchase` chi co 0/1/null.
- Null label chi xuat hien khi `has_full_30d_label_window = 0`.
- `future_30d_revenue >= 0`.
- Feature categorical chinh khong con placeholder tho/null.
- Du lieu goc `train_sessions` va `test_sessions` van duoc giu nguyen.